# Geopolitic GNN — training (Kaggle)

Self-contained: upload **two** things to Kaggle and *Run All* —
1. the `dataset_parquet/` folder as a **Dataset** (Add Input), and
2. this notebook.

Set **Accelerator = GPU**, **Internet = On**, and add a **Secret** `WANDB_API_KEY`.
Outputs (best.pt, preprocess.pkl, calibrator.pkl, metrics.json) land in `/kaggle/working/artifacts` and stream to W&B (`anna-nechytailenko-kyiv-school-of-economics/geosimulation`).

In [ ]:
# Kaggle GPU images ship torch, numpy, pandas, scikit-learn, pyarrow, tqdm.
!pip -q install torch-geometric torchmetrics captum wandb tqdm


In [ ]:
import base64, pathlib
pathlib.Path('ml').mkdir(exist_ok=True)
FILES = {
"ml/config.py": "IiIiQ2VudHJhbCBjb25maWd1cmF0aW9uIGZvciB0aGUgTUwgc3RhZ2UuIERlcGVuZGVuY3ktZnJlZSAoc3RkbGliIG9ubHkpIHNvIGl0IGlzCmltcG9ydGFibGUgb24gYW55IG1hY2hpbmUgYW5kIGluIHVuaXQgdGVzdHMgd2l0aG91dCB0b3JjaCBpbnN0YWxsZWQuCgpDTEFTU19OQU1FUyBpcyB0aGUgc2luZ2xlIHNvdXJjZSBvZiB0cnV0aCBvbiB0aGUgUHl0aG9uIHNpZGUgYW5kIE1VU1Qgc3RheSBieXRlLWZvci1ieXRlCmlkZW50aWNhbCB0byBgaW50ZXJuYWwvbGFiZWwuQ2xhc3Nlc2AgaW4gdGhlIEdvIGNvZGUgKHRoZSBpbmRleCBvZiBlYWNoIGNsYXNzIGlzIHBlcnNpc3RlZAppbiBgY2xhc3NfZGlzdHJpYnV0aW9uYCBvbiBldmVyeSBTTkFQU0hPVF9FREdFKS4gdGVzdF9jb25maWcucHkgYXNzZXJ0cyB0aGlzIGFnYWluc3QgdGhlIEdvCnNvdXJjZSBzbyB0aGUgdHdvIGNhbiBuZXZlciBzaWxlbnRseSBkcmlmdC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgb3MKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZCwgYXNkaWN0CmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCiMgLS0tLSBjYW5vbmljYWwgY2xhc3Mgb3JkZXIgKG1pcnJvciBvZiBpbnRlcm5hbC9sYWJlbC9jYW1lby5nbyBgQ2xhc3Nlc2ApIC0tLS0tLS0tLS0tLS0KQ0xBU1NfTkFNRVM6IGxpc3Rbc3RyXSA9IFsKICAgICJNQVRFUklBTF9DT05GTElDVCIsCiAgICAiVkVSQkFMX0NPTkZMSUNUIiwKICAgICJNQVRFUklBTF9DT09QRVJBVElPTiIsCiAgICAiVkVSQkFMX0NPT1BFUkFUSU9OIiwKICAgICJTVEFUVVNfUVVPIiwKXQpOVU1fQ0xBU1NFUyA9IGxlbihDTEFTU19OQU1FUykKU1RBVFVTX1FVT19JTkRFWCA9IENMQVNTX05BTUVTLmluZGV4KCJTVEFUVVNfUVVPIikKCgpkZWYgY2xhc3NfaW5kZXgobmFtZTogc3RyKSAtPiBpbnQ6CiAgICAiIiJDYW5vbmljYWwgaW5kZXggb2YgYSBjbGFzcyBuYW1lLCBvciAtMSBpZiB1bmtub3duIChtaXJyb3JzIGxhYmVsLkluZGV4KS4iIiIKICAgIHRyeToKICAgICAgICByZXR1cm4gQ0xBU1NfTkFNRVMuaW5kZXgobmFtZSkKICAgIGV4Y2VwdCBWYWx1ZUVycm9yOgogICAgICAgIHJldHVybiAtMQoKCmRlZiBfZW52X2ludChrZXk6IHN0ciwgZGVmYXVsdDogaW50KSAtPiBpbnQ6CiAgICB2ID0gb3MuZW52aXJvbi5nZXQoa2V5KQogICAgcmV0dXJuIGludCh2KSBpZiB2IG5vdCBpbiAoTm9uZSwgIiIpIGVsc2UgZGVmYXVsdAoKCmRlZiBfZW52X2Zsb2F0KGtleTogc3RyLCBkZWZhdWx0OiBmbG9hdCkgLT4gZmxvYXQ6CiAgICB2ID0gb3MuZW52aXJvbi5nZXQoa2V5KQogICAgcmV0dXJuIGZsb2F0KHYpIGlmIHYgbm90IGluIChOb25lLCAiIikgZWxzZSBkZWZhdWx0CgoKQGRhdGFjbGFzcwpjbGFzcyBDb25maWc6CiAgICAiIiJBbGwga25vYnMgZm9yIGV4cG9ydCDihpIgZGF0YXNldCDihpIgdHJhaW4g4oaSIGNhbGlicmF0ZSDihpIgc2VydmUuCgogICAgVGltZSBjb252ZW50aW9uIChtaXJyb3Igb2YgaW50ZXJuYWwvdGltZXN0ZXApOiB0aW1lX3N0ZXAgPSAoeWVhci0yMDEwKSoxMiArIChtb250aC0xKSwKICAgIHQ9MCAtPiAyMDEwLTAxLiBUaGUgYWdncmVnYXRlZCBEQiBzcGFucyB0cyAwLi5tYXhfdHMgKDE5NyA9IDIwMjYtMDYpLgogICAgIiIiCgogICAgIyAtLS0tIHBhdGhzIC0tLS0KICAgIGRhdGFfZGlyOiBzdHIgPSAiZGF0YXNldF9wYXJxdWV0IiAgICAgIyBob2xkcyBub2RlX3NuYXBzaG90cy8gc25hcHNob3RfZWRnZXMvIHN0cnVjdHVyYWxfZWRnZXMucGFycXVldAogICAgYXJ0aWZhY3RzX2Rpcjogc3RyID0gImFydGlmYWN0cyIgICAgICAjIGJlc3QucHQgLyBsYXN0LnB0IC8gcHJlcHJvY2Vzcy5wa2wgLyBjYWxpYnJhdG9yLnBrbCAvIG1ldHJpY3MuanNvbgoKICAgICMgLS0tLSB0aW1lIC8gd2luZG93aW5nIC0tLS0KICAgIHdpbmRvdzogaW50ID0gMTIgICAgICAgICAgICAgICAgICAgICAgIyBXOiByb2xsaW5nIGlucHV0IHdpbmRvdyBpbiBtb250aHMKICAgIG1heF90czogaW50ID0gMTk3ICAgICAgICAgICAgICAgICAgICAgIyBNYXhUaW1lU3RlcCBwcmVzZW50IGluIHRoZSBhZ2dyZWdhdGVkIERCICgyMDI2LTA2KQogICAgbWluX3RhcmdldF90czogaW50ID0gMTEgICAgICAgICAgICAgICAjIHNtYWxsZXN0IFQgd2l0aCBhIGZ1bGwgMTItbW9udGggd2luZG93CiAgICAjIGNocm9ub2xvZ2ljYWwgc3BsaXQgYnkgVEFSR0VUIG1vbnRoIFQgKGxhYmVsIGlzIGF0IFQrMSk7IHNlZSBwbGFucy8wMyDCpzEuNgogICAgdHJhaW5fbWF4X3RzOiBpbnQgPSAxNzIgICAgICAgICAgICAgICAjIHRyYWluOiAxMS4uMTcyCiAgICB2YWxfbWF4X3RzOiBpbnQgPSAxODQgICAgICAgICAgICAgICAgICMgdmFsOiAgIDE3My4uMTg0CiAgICAjIHRlc3Q6ICh2YWxfbWF4X3RzKzEpIC4uIChtYXhfdHMtMSkgPSAxODUuLjE5NiAgKFQrMSBtdXN0IGJlIDw9IG1heF90cykKCiAgICAjIC0tLS0gc2FtcGxpbmcgLyBpbWJhbGFuY2UgLS0tLQogICAga19uZWc6IGludCA9IDUgICAgICAgICAgICAgICAgICAgICAgICAjIFNUQVRVU19RVU8gbmVnYXRpdmVzIHBlciBwb3NpdGl2ZSBDb3VudHJ5LT5Db3VudHJ5IGVkZ2UKICAgIGxvc3M6IHN0ciA9ICJ3ZWlnaHRlZF9jZSIgICAgICAgICAgICAjICJ3ZWlnaHRlZF9jZSIgfCAiZm9jYWwiCiAgICBmb2NhbF9nYW1tYTogZmxvYXQgPSAyLjAKCiAgICAjIC0tLS0gbW9kZWwgLS0tLQogICAgaGlkZGVuOiBpbnQgPSAxMjggICAgICAgICAgICAgICAgICAgICAjIGQKICAgIGhlYWRzOiBpbnQgPSA0ICAgICAgICAgICAgICAgICAgICAgICAgIyBIIChhdHRlbnRpb24gaGVhZHMpOyBoaWRkZW4gbXVzdCBiZSBkaXZpc2libGUgYnkgaGVhZHMKICAgIGlkX2RpbTogaW50ID0gNjQgICAgICAgICAgICAgICAgICAgICAgIyBpZGVudGl0eSBlbWJlZGRpbmcgd2lkdGgKICAgIGhvcHM6IGludCA9IDIgICAgICAgICAgICAgICAgICAgICAgICAgIyBzcGF0aWFsIG1lc3NhZ2UtcGFzc2luZyBob3BzCiAgICBncnVfbGF5ZXJzOiBpbnQgPSAxCiAgICBkcm9wb3V0OiBmbG9hdCA9IDAuMwoKICAgICMgLS0tLSBvcHRpbWl6YXRpb24gLS0tLQogICAgbHI6IGZsb2F0ID0gMWUtMwogICAgd2VpZ2h0X2RlY2F5OiBmbG9hdCA9IDFlLTQKICAgIGVwb2NoczogaW50ID0gNTAKICAgIHBhdGllbmNlOiBpbnQgPSA1ICAgICAgICAgICAgICAgICAgICAgIyBlYXJseS1zdG9wcGluZyBwYXRpZW5jZSBvbiB2YWwgbWFjcm8tRjEKICAgIHNlZWQ6IGludCA9IDQyCiAgICBkZXZpY2U6IHN0ciA9ICJhdXRvIiAgICAgICAgICAgICAgICAgICMgImF1dG8iIHwgImNwdSIgfCAiY3VkYSIKCiAgICAjIC0tLS0gZXhwZXJpbWVudCB0cmFja2luZyAoV2VpZ2h0cyAmIEJpYXNlcykgLS0tLQogICAgd2FuZGJfZW50aXR5OiBzdHIgPSAiYW5uYS1uZWNoeXRhaWxlbmtvLWt5aXYtc2Nob29sLW9mLWVjb25vbWljcyIKICAgIHdhbmRiX3Byb2plY3Q6IHN0ciA9ICJnZW9zaW11bGF0aW9uIgogICAgd2FuZGJfbW9kZTogc3RyID0gIm9ubGluZSIgICAgICAgICAgICAjICJvbmxpbmUiIHwgIm9mZmxpbmUiIHwgImRpc2FibGVkIgogICAgcnVuX25hbWU6IHN0ciA9ICIiCgogICAgY2xhc3NfbmFtZXM6IGxpc3Rbc3RyXSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1sYW1iZGE6IGxpc3QoQ0xBU1NfTkFNRVMpKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgQHByb3BlcnR5CiAgICBkZWYgdGVzdF9taW5fdHMoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBzZWxmLnZhbF9tYXhfdHMgKyAxCgogICAgQHByb3BlcnR5CiAgICBkZWYgdGVzdF9tYXhfdHMoc2VsZikgLT4gaW50OgogICAgICAgICMgVCsxIG11c3QgZXhpc3QgKDw9IG1heF90cyksIHNvIHRoZSBsYXN0IHVzYWJsZSB0YXJnZXQgbW9udGggaXMgbWF4X3RzLTEuCiAgICAgICAgcmV0dXJuIHNlbGYubWF4X3RzIC0gMQoKICAgIGRlZiBzcGxpdF9vZihzZWxmLCB0YXJnZXRfdHM6IGludCkgLT4gc3RyOgogICAgICAgICIiIlJldHVybiAndHJhaW4nIHwgJ3ZhbCcgfCAndGVzdCcgZm9yIGEgdGFyZ2V0IG1vbnRoIFQgKG9yICdub25lJykuIiIiCiAgICAgICAgaWYgdGFyZ2V0X3RzIDwgc2VsZi5taW5fdGFyZ2V0X3RzIG9yIHRhcmdldF90cyA+IHNlbGYudGVzdF9tYXhfdHM6CiAgICAgICAgICAgIHJldHVybiAibm9uZSIKICAgICAgICBpZiB0YXJnZXRfdHMgPD0gc2VsZi50cmFpbl9tYXhfdHM6CiAgICAgICAgICAgIHJldHVybiAidHJhaW4iCiAgICAgICAgaWYgdGFyZ2V0X3RzIDw9IHNlbGYudmFsX21heF90czoKICAgICAgICAgICAgcmV0dXJuICJ2YWwiCiAgICAgICAgcmV0dXJuICJ0ZXN0IgoKICAgIGRlZiB0YXJnZXRfbW9udGhzKHNlbGYsIHNwbGl0OiBzdHIpIC0+IGxpc3RbaW50XToKICAgICAgICByZXR1cm4gW3QgZm9yIHQgaW4gcmFuZ2Uoc2VsZi5taW5fdGFyZ2V0X3RzLCBzZWxmLnRlc3RfbWF4X3RzICsgMSkgaWYgc2VsZi5zcGxpdF9vZih0KSA9PSBzcGxpdF0KCiAgICBkZWYgdG9fZGljdChzZWxmKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICAgICByZXR1cm4gYXNkaWN0KHNlbGYpCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgZnJvbV9lbnYoY2xzKSAtPiAiQ29uZmlnIjoKICAgICAgICAiIiJCdWlsZCBhIENvbmZpZywgb3ZlcnJpZGluZyBzZWxlY3RlZCBmaWVsZHMgZnJvbSBlbnZpcm9ubWVudCB2YXJpYWJsZXMuCgogICAgICAgIE9uIEthZ2dsZSB0aGUgZGF0YSBsaXZlcyB1bmRlciAva2FnZ2xlL2lucHV0LzxkYXRhc2V0Pi87IHNldCBHRU9fREFUQV9ESVIgdGhlcmUuCiAgICAgICAgIiIiCiAgICAgICAgYyA9IGNscygpCiAgICAgICAgYy5kYXRhX2RpciA9IG9zLmVudmlyb24uZ2V0KCJHRU9fREFUQV9ESVIiLCBjLmRhdGFfZGlyKQogICAgICAgIGMuYXJ0aWZhY3RzX2RpciA9IG9zLmVudmlyb24uZ2V0KCJHRU9fQVJUSUZBQ1RTX0RJUiIsIGMuYXJ0aWZhY3RzX2RpcikKICAgICAgICBjLmVwb2NocyA9IF9lbnZfaW50KCJHRU9fRVBPQ0hTIiwgYy5lcG9jaHMpCiAgICAgICAgYy5wYXRpZW5jZSA9IF9lbnZfaW50KCJHRU9fUEFUSUVOQ0UiLCBjLnBhdGllbmNlKQogICAgICAgIGMuc2VlZCA9IF9lbnZfaW50KCJHRU9fU0VFRCIsIGMuc2VlZCkKICAgICAgICBjLmtfbmVnID0gX2Vudl9pbnQoIkdFT19LX05FRyIsIGMua19uZWcpCiAgICAgICAgYy5sciA9IF9lbnZfZmxvYXQoIkdFT19MUiIsIGMubHIpCiAgICAgICAgYy5sb3NzID0gb3MuZW52aXJvbi5nZXQoIkdFT19MT1NTIiwgYy5sb3NzKQogICAgICAgIGMuZGV2aWNlID0gb3MuZW52aXJvbi5nZXQoIkdFT19ERVZJQ0UiLCBjLmRldmljZSkKICAgICAgICBjLndhbmRiX2VudGl0eSA9IG9zLmVudmlyb24uZ2V0KCJXQU5EQl9FTlRJVFkiLCBjLndhbmRiX2VudGl0eSkKICAgICAgICBjLndhbmRiX3Byb2plY3QgPSBvcy5lbnZpcm9uLmdldCgiV0FOREJfUFJPSkVDVCIsIGMud2FuZGJfcHJvamVjdCkKICAgICAgICBjLndhbmRiX21vZGUgPSBvcy5lbnZpcm9uLmdldCgiV0FOREJfTU9ERSIsIGMud2FuZGJfbW9kZSkKICAgICAgICBjLnJ1bl9uYW1lID0gb3MuZW52aXJvbi5nZXQoIldBTkRCX1JVTl9OQU1FIiwgYy5ydW5fbmFtZSkKICAgICAgICByZXR1cm4gYwo=",
"ml/timestep.py": "IiIiUHl0aG9uIG1pcnJvciBvZiBpbnRlcm5hbC90aW1lc3RlcC90aW1lc3RlcC5nby4gRGVwZW5kZW5jeS1mcmVlLgoKdGltZV9zdGVwID0gKHllYXIgLSAyMDEwKSAqIDEyICsgKG1vbnRoIC0gMSksIHQ9MCAtPiAyMDEwLTAxLiBDYWxlbmRhciBmaWVsZHMgYXJlIHB1cmUKZnVuY3Rpb25zIG9mIHRpbWVfc3RlcCAobm8gY2FsZW5kYXIgbm9kZSBhbnl3aGVyZSkuIFVzZXMgUHl0aG9uJ3MgZmxvb3JpbmcgLy8sJSB3aGljaAphbHJlYWR5IG1hdGNoIEdvJ3MgZmxvb3JEaXYvZmxvb3JNb2QgZm9yIG5lZ2F0aXZlIGlucHV0cy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpFUE9DSCA9IDIwMTAgICMgY2FsZW5kYXIgeWVhciB0aGF0IG1hcHMgdG8gdGltZV9zdGVwIDAgKEphbnVhcnkpCgoKZGVmIGZyb21feW0oeWVhcjogaW50LCBtb250aDogaW50KSAtPiBpbnQ6CiAgICAiIiIoeWVhciwgbW9udGhbMS0xMl0pIC0+IHRpbWVfc3RlcC4iIiIKICAgIHJldHVybiAoeWVhciAtIEVQT0NIKSAqIDEyICsgKG1vbnRoIC0gMSkKCgpkZWYgZnJvbV95ZWFyKHllYXI6IGludCkgLT4gaW50OgogICAgIiIiQ2FsZW5kYXIgeWVhciAtPiB0aW1lX3N0ZXAgb2YgaXRzIEphbnVhcnkuIiIiCiAgICByZXR1cm4gZnJvbV95bSh5ZWFyLCAxKQoKCmRlZiB5ZWFyKHRzOiBpbnQpIC0+IGludDoKICAgIHJldHVybiBFUE9DSCArICh0cyAvLyAxMikKCgpkZWYgbW9udGgodHM6IGludCkgLT4gaW50OgogICAgcmV0dXJuICh0cyAlIDEyKSArIDEKCgpkZWYgaXNvX3BlcmlvZCh0czogaW50KSAtPiBzdHI6CiAgICAiIiInWVlZWS1NTScgZm9yIGEgdGltZV9zdGVwIChlLmcuICcyMDEzLTA3JykuIiIiCiAgICByZXR1cm4gZiJ7eWVhcih0cyk6MDRkfS17bW9udGgodHMpOjAyZH0iCgoKZGVmIGNsYW1wX3N0YXJ0KHRzOiBpbnQpIC0+IGludDoKICAgICIiIlN0cnVjdHVyYWwgZWRnZXMgdGhhdCBiZWdhbiBiZWZvcmUgMjAxMCBjbGFtcCB0byBzdGFydF90aW1lX3N0ZXAgPSAwLiIiIgogICAgcmV0dXJuIDAgaWYgdHMgPCAwIGVsc2UgdHMK",
"ml/features.py": "IiIiRmVhdHVyZSBlbmdpbmVlcmluZyAmIHRoZSBwZXJzaXN0ZWQgcHJlcHJvY2Vzc2luZyBidW5kbGUgKHBsYW5zLzAzIMKnMS40LCDCpzEuNykuCgpgUHJlcHJvY2Vzc2AgaXMgZml0IG9uIHRoZSBUUkFJTiBtb250aHMgb25seSAodGhlICMxIGxlYWthZ2UgZ3VhcmRyYWlsKSwgdGhlbiBhcHBsaWVkIHRvIGFsbApzcGxpdHMgYW5kIHJldXNlZCB1bmNoYW5nZWQgYXQgaW5mZXJlbmNlLiBJdCBpcyBzZXJpYWxpemVkIGFzIHBhcnQgb2YgYHByZXByb2Nlc3MucGtsYCBzbyBhCnNlcnZlZCBtb2RlbCBjYW4gbmV2ZXIgZHJpZnQgZnJvbSBob3cgaXQgd2FzIHRyYWluZWQuIEl0IG93bnM6IHBlci1ub2RlLXR5cGUgY29udGludW91cwpzY2FsZXJzICsgaW1wdXRlIG1lYW5zLCB0aGUgcmVnaW9uIG9uZS1ob3Qgdm9jYWJ1bGFyeSwgdGhlIGFjdG9yLWlkIGNvbHVtbiBtYXAgKGFsc28gdGhlCmFsbGlhbmNlIG11bHRpLWhvdCBjb2x1bW5zKSwgdGhlIGZpeGVkIG5vZGUgb3JkZXJpbmcsIGFuZCB0aGUgZWRnZS1mZWF0dXJlIHNjYWxlci4KClRoZSBhbGxpYW5jZSBtdWx0aS1ob3QgaXRzZWxmIGlzIGJ1aWx0IGluIGRhdGFzZXQucHkgKGl0IG5lZWRzIHRoZSB0ZW1wb3JhbCBNRU1CRVJfT0YKdmFsaWRpdHksIHdoaWNoIGlzIGdyYXBoIGRhdGEpOyB0aGlzIG1vZHVsZSBvbmx5IG93bnMgdGhlIGNvbHVtbiBvcmRlcmluZyAoYGFjdG9yX2lkc2ApLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCBqb2JsaWIKZnJvbSBza2xlYXJuLnByZXByb2Nlc3NpbmcgaW1wb3J0IFN0YW5kYXJkU2NhbGVyCgojIC0tLS0gZmVhdHVyZSBncm91cHMgKG5hbWVzIHZlcmlmaWVkIGFnYWluc3QgdGhlIEdvIGxvYWRlcnMpIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCkNPVU5UUllfQ09OVCA9IFsKICAgICJnZHBfbG9nIiwgImdkcF9wZXJfY2FwaXRhIiwgInRyYWRlX29wZW5uZXNzX2luZGV4IiwgInBvcHVsYXRpb25fbG9nIiwKICAgICJwb3B1bGF0aW9uX2dyb3d0aCIsICJsYW5kX2FyZWFfbG9nIiwgInBvbGl0aWNhbF9zdGFiaWxpdHkiLCAidmRlbV9wb2x5YXJjaHlfc2NvcmUiLAogICAgInllYXJzX3NpbmNlX2xlYWRlcnNoaXBfY2hhbmdlIiwgIm1pbGl0YXJ5X2V4cGVuZGl0dXJlX2xvZyIsICJoZGkiLAogICAgImFjdGl2ZV9jb25mbGljdF9jb3VudCIsICJjb25mbGljdF9pbnRlbnNpdHkiLApdCkNPVU5UUllfTE9HMVAgPSBbImFjdGl2ZV9jb25mbGljdF9jb3VudCJdICAjICpfbG9nICsgY29uZmxpY3RfaW50ZW5zaXR5IGFyZSBhbHJlYWR5IGxvZy1zY2FsZWQKQ09VTlRSWV9CSU4gPSBbInNhbmN0aW9uc19zdGF0dXMiLCAidW5zY19zZWF0X2ZsYWciLCAibnVjbGVhcl9mbGFnIiwgImNvYXN0bGluZV9mbGFnIl0KCkFDVE9SX0NPTlQgPSBbIm1lbWJlcl9jb3VudF9sb2ciLCAicmVjb2duaXplZF9sZWdpdGltYWN5X3Njb3JlIiwgImZpbmFuY2lhbF9yZXNvdXJjZXNfdGllciJdCkFDVE9SX0xPRzFQID0gWyJyZWNvZ25pemVkX2xlZ2l0aW1hY3lfc2NvcmUiXQoKRURHRV9DT05UID0gWyJldmVudF9jb3VudCIsICJ3ZWlnaHRlZF9pbnRlbnNpdHkiLCAic2VudGltZW50X21lYW4iLCAic2VudGltZW50X3N0ZCIsICJkYXlzX3NpbmNlX2xhc3RfZXZlbnQiXQpFREdFX0xPRzFQID0gWyJldmVudF9jb3VudCIsICJkYXlzX3NpbmNlX2xhc3RfZXZlbnQiXQpFREdFX0RJU1RfTEVOID0gNSAgIyBjbGFzc19kaXN0cmlidXRpb25bNV0KCgpkZWYgX2FwcGx5X2xvZzFwKGRmOiBwZC5EYXRhRnJhbWUsIGNvbHM6IGxpc3Rbc3RyXSkgLT4gcGQuRGF0YUZyYW1lOgogICAgb3V0ID0gZGYuY29weSgpCiAgICBmb3IgYyBpbiBjb2xzOgogICAgICAgIGlmIGMgaW4gb3V0LmNvbHVtbnM6CiAgICAgICAgICAgIG91dFtjXSA9IG5wLmxvZzFwKG91dFtjXS5hc3R5cGUoImZsb2F0NjQiKS5jbGlwKGxvd2VyPTApKQogICAgcmV0dXJuIG91dAoKCmRlZiBfc3RhY2tfZGlzdHJpYnV0aW9uKHNlcmllczogcGQuU2VyaWVzKSAtPiBucC5uZGFycmF5OgogICAgIiIiVHVybiBhIHBhcnF1ZXQgbGlzdDxmbG9hdD5bNV0gY29sdW1uIGludG8gYSBbTiwgNV0gZmxvYXQgYXJyYXkgKHJvYnVzdCB0byBOb25lKS4iIiIKICAgIHJvd3MgPSBbXQogICAgZm9yIHYgaW4gc2VyaWVzLnRvbGlzdCgpOgogICAgICAgIGlmIHYgaXMgTm9uZSBvciAoaXNpbnN0YW5jZSh2LCBmbG9hdCkgYW5kIG5wLmlzbmFuKHYpKToKICAgICAgICAgICAgcm93cy5hcHBlbmQoWzAuMF0gKiBFREdFX0RJU1RfTEVOKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGFyciA9IGxpc3QodikKICAgICAgICAgICAgYXJyID0gKGFyciArIFswLjBdICogRURHRV9ESVNUX0xFTilbOkVER0VfRElTVF9MRU5dCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKFtmbG9hdCh4KSBmb3IgeCBpbiBhcnJdKQogICAgcmV0dXJuIG5wLmFzYXJyYXkocm93cywgZHR5cGU9ImZsb2F0MzIiKQoKCkBkYXRhY2xhc3MKY2xhc3MgUHJlcHJvY2VzczoKICAgIGNvdW50cnlfaWRzOiBsaXN0W3N0cl0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkgICAjIGZpeGVkIENvdW50cnkgbm9kZSBvcmRlcmluZwogICAgYWN0b3JfaWRzOiBsaXN0W3N0cl0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkgICAgICMgZml4ZWQgQWN0b3Igb3JkZXJpbmcgPSBhbGxpYW5jZSBjb2x1bW5zCiAgICByZWdpb25zOiBsaXN0W3N0cl0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkgICAgICAgIyByZWdpb24gb25lLWhvdCB2b2NhYnVsYXJ5CiAgICBjbGFzc19uYW1lczogbGlzdFtzdHJdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCgogICAgY291bnRyeV9tZWFuczogZGljdFtzdHIsIGZsb2F0XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQogICAgYWN0b3JfbWVhbnM6IGRpY3Rbc3RyLCBmbG9hdF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIGVkZ2VfbWVhbnM6IGRpY3Rbc3RyLCBmbG9hdF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIGNvdW50cnlfc2NhbGVyOiBTdGFuZGFyZFNjYWxlciB8IE5vbmUgPSBOb25lCiAgICBhY3Rvcl9zY2FsZXI6IFN0YW5kYXJkU2NhbGVyIHwgTm9uZSA9IE5vbmUKICAgIGVkZ2Vfc2NhbGVyOiBTdGFuZGFyZFNjYWxlciB8IE5vbmUgPSBOb25lCgogICAgIyAtLS0tIGRpbXMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIEBwcm9wZXJ0eQogICAgZGVmIGFsbGlhbmNlX2RpbShzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLmFjdG9yX2lkcykKCiAgICBAcHJvcGVydHkKICAgIGRlZiBjb3VudHJ5X2Jsb2NrX2RpbShzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihDT1VOVFJZX0NPTlQpICsgbGVuKENPVU5UUllfQklOKSArIGxlbihzZWxmLnJlZ2lvbnMpCgogICAgQHByb3BlcnR5CiAgICBkZWYgY291bnRyeV9mZWF0X2RpbShzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIHNlbGYuY291bnRyeV9ibG9ja19kaW0gKyBzZWxmLmFsbGlhbmNlX2RpbSAgIyArIGFsbGlhbmNlIG11bHRpLWhvdCAoYWRkZWQgaW4gZGF0YXNldCkKCiAgICBAcHJvcGVydHkKICAgIGRlZiBhY3Rvcl9mZWF0X2RpbShzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihBQ1RPUl9DT05UKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGVkZ2VfZGltKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKEVER0VfQ09OVCkgKyBFREdFX0RJU1RfTEVOCgogICAgIyAtLS0tIGZpdCAoVFJBSU4gbW9udGhzIG9ubHkpIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBmaXQoc2VsZiwgbm9kZV9kZjogcGQuRGF0YUZyYW1lLCBlZGdlX2RmOiBwZC5EYXRhRnJhbWUsIHRyYWluX21heF90czogaW50KSAtPiAiUHJlcHJvY2VzcyI6CiAgICAgICAgY291bnRyaWVzID0gbm9kZV9kZltub2RlX2RmWyJub2RlX3R5cGUiXSA9PSAiQ291bnRyeSJdCiAgICAgICAgYWN0b3JzID0gbm9kZV9kZltub2RlX2RmWyJub2RlX3R5cGUiXSA9PSAiQWN0b3IiXQoKICAgICAgICBzZWxmLmNvdW50cnlfaWRzID0gc29ydGVkKGNvdW50cmllc1sibm9kZV9pZCJdLnVuaXF1ZSgpLnRvbGlzdCgpKQogICAgICAgIHNlbGYuYWN0b3JfaWRzID0gc29ydGVkKGFjdG9yc1sibm9kZV9pZCJdLnVuaXF1ZSgpLnRvbGlzdCgpKQogICAgICAgIHNlbGYucmVnaW9ucyA9IHNvcnRlZCh4IGZvciB4IGluIGNvdW50cmllc1sicmVnaW9uIl0uZHJvcG5hKCkudW5pcXVlKCkudG9saXN0KCkgaWYgc3RyKHgpICE9ICIiKQoKICAgICAgICBjX3RyID0gX2FwcGx5X2xvZzFwKGNvdW50cmllc1tjb3VudHJpZXNbInRzIl0gPD0gdHJhaW5fbWF4X3RzXSwgQ09VTlRSWV9MT0cxUCkKICAgICAgICBzZWxmLmNvdW50cnlfbWVhbnMgPSB7YzogZmxvYXQoY190cltjXS5tZWFuKCkpIGZvciBjIGluIENPVU5UUllfQ09OVCBpZiBjIGluIGNfdHJ9CiAgICAgICAgc2VsZi5jb3VudHJ5X3NjYWxlciA9IFN0YW5kYXJkU2NhbGVyKCkuZml0KHNlbGYuX2NvbnRfbWF0cml4KGNfdHIsIENPVU5UUllfQ09OVCwgc2VsZi5jb3VudHJ5X21lYW5zKSkKCiAgICAgICAgYV90ciA9IF9hcHBseV9sb2cxcChhY3RvcnNbYWN0b3JzWyJ0cyJdIDw9IHRyYWluX21heF90c10sIEFDVE9SX0xPRzFQKQogICAgICAgIHNlbGYuYWN0b3JfbWVhbnMgPSB7YzogZmxvYXQoYV90cltjXS5tZWFuKCkpIGZvciBjIGluIEFDVE9SX0NPTlQgaWYgYyBpbiBhX3RyfQogICAgICAgIHNlbGYuYWN0b3Jfc2NhbGVyID0gU3RhbmRhcmRTY2FsZXIoKS5maXQoc2VsZi5fY29udF9tYXRyaXgoYV90ciwgQUNUT1JfQ09OVCwgc2VsZi5hY3Rvcl9tZWFucykpCgogICAgICAgIGVfdHIgPSBfYXBwbHlfbG9nMXAoZWRnZV9kZltlZGdlX2RmWyJ0cyJdIDw9IHRyYWluX21heF90c10sIEVER0VfTE9HMVApCiAgICAgICAgc2VsZi5lZGdlX21lYW5zID0ge2M6IGZsb2F0KGVfdHJbY10ubWVhbigpKSBmb3IgYyBpbiBFREdFX0NPTlQgaWYgYyBpbiBlX3RyfQogICAgICAgIHNlbGYuZWRnZV9zY2FsZXIgPSBTdGFuZGFyZFNjYWxlcigpLmZpdChzZWxmLl9jb250X21hdHJpeChlX3RyLCBFREdFX0NPTlQsIHNlbGYuZWRnZV9tZWFucykpCiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICAjIC0tLS0gdHJhbnNmb3JtcyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIF9jb250X21hdHJpeChkZjogcGQuRGF0YUZyYW1lLCBjb2xzOiBsaXN0W3N0cl0sIG1lYW5zOiBkaWN0W3N0ciwgZmxvYXRdKSAtPiBucC5uZGFycmF5OgogICAgICAgIG91dCA9IG5wLmVtcHR5KChsZW4oZGYpLCBsZW4oY29scykpLCBkdHlwZT0iZmxvYXQ2NCIpCiAgICAgICAgZm9yIGosIGMgaW4gZW51bWVyYXRlKGNvbHMpOgogICAgICAgICAgICBjb2wgPSBkZltjXS5hc3R5cGUoImZsb2F0NjQiKSBpZiBjIGluIGRmLmNvbHVtbnMgZWxzZSBwZC5TZXJpZXMobnAubmFuLCBpbmRleD1kZi5pbmRleCkKICAgICAgICAgICAgb3V0WzosIGpdID0gY29sLmZpbGxuYShtZWFucy5nZXQoYywgMC4wKSkudG9fbnVtcHkoKQogICAgICAgIHJldHVybiBvdXQKCiAgICBkZWYgX29uZWhvdF9yZWdpb24oc2VsZiwgcmVnaW9uczogcGQuU2VyaWVzKSAtPiBucC5uZGFycmF5OgogICAgICAgIGlkeCA9IHtyOiBpIGZvciBpLCByIGluIGVudW1lcmF0ZShzZWxmLnJlZ2lvbnMpfQogICAgICAgIG91dCA9IG5wLnplcm9zKChsZW4ocmVnaW9ucyksIGxlbihzZWxmLnJlZ2lvbnMpKSwgZHR5cGU9ImZsb2F0MzIiKQogICAgICAgIGZvciBpLCByIGluIGVudW1lcmF0ZShyZWdpb25zLnRvbGlzdCgpKToKICAgICAgICAgICAgaiA9IGlkeC5nZXQocikKICAgICAgICAgICAgaWYgaiBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIG91dFtpLCBqXSA9IDEuMAogICAgICAgIHJldHVybiBvdXQKCiAgICBkZWYgX3JlaW5kZXgoc2VsZiwgZGZfbW9udGg6IHBkLkRhdGFGcmFtZSwgaWRzOiBsaXN0W3N0cl0pIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICByZXR1cm4gZGZfbW9udGguc2V0X2luZGV4KCJub2RlX2lkIikucmVpbmRleChpZHMpCgogICAgZGVmIGNvdW50cnlfYmxvY2soc2VsZiwgZGZfbW9udGg6IHBkLkRhdGFGcmFtZSkgLT4gbnAubmRhcnJheToKICAgICAgICAiIiJbTmMsIGNvdW50cnlfYmxvY2tfZGltXSA9IHNjYWxlZCBjb250aW51b3VzIHwgYmluYXJ5IHwgcmVnaW9uIG9uZS1ob3QuIE9yZGVyZWQgYnkKICAgICAgICBjb3VudHJ5X2lkczsgcm93cyBhYnNlbnQgaW4gZGZfbW9udGggYXJlIGltcHV0ZWQgKG1lYW4gZm9yIGNvbnRpbnVvdXMsIDAgZm9yIGZsYWdzKS4iIiIKICAgICAgICBkZiA9IHNlbGYuX3JlaW5kZXgoZGZfbW9udGgsIHNlbGYuY291bnRyeV9pZHMpCiAgICAgICAgY29udCA9IF9hcHBseV9sb2cxcChkZiwgQ09VTlRSWV9MT0cxUCkKICAgICAgICBjb250ID0gc2VsZi5jb3VudHJ5X3NjYWxlci50cmFuc2Zvcm0oc2VsZi5fY29udF9tYXRyaXgoY29udCwgQ09VTlRSWV9DT05ULCBzZWxmLmNvdW50cnlfbWVhbnMpKQogICAgICAgIGJpbm0gPSBucC56ZXJvcygobGVuKGRmKSwgbGVuKENPVU5UUllfQklOKSksIGR0eXBlPSJmbG9hdDMyIikKICAgICAgICBmb3IgaiwgYyBpbiBlbnVtZXJhdGUoQ09VTlRSWV9CSU4pOgogICAgICAgICAgICBpZiBjIGluIGRmLmNvbHVtbnM6CiAgICAgICAgICAgICAgICBiaW5tWzosIGpdID0gZGZbY10uZmlsbG5hKDApLmFzdHlwZSgiZmxvYXQzMiIpLnRvX251bXB5KCkKICAgICAgICByZWdpb24gPSBzZWxmLl9vbmVob3RfcmVnaW9uKGRmWyJyZWdpb24iXSBpZiAicmVnaW9uIiBpbiBkZi5jb2x1bW5zIGVsc2UgcGQuU2VyaWVzKFtOb25lXSAqIGxlbihkZikpKQogICAgICAgIHJldHVybiBucC5oc3RhY2soW2NvbnQuYXN0eXBlKCJmbG9hdDMyIiksIGJpbm0sIHJlZ2lvbl0pLmFzdHlwZSgiZmxvYXQzMiIpCgogICAgZGVmIGFjdG9yX2Jsb2NrKHNlbGYsIGRmX21vbnRoOiBwZC5EYXRhRnJhbWUpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgIiIiW05hLCBhY3Rvcl9mZWF0X2RpbV0gPSBzY2FsZWQgYWN0b3IgY29udGludW91cyBmZWF0dXJlcywgb3JkZXJlZCBieSBhY3Rvcl9pZHMuIiIiCiAgICAgICAgZGYgPSBzZWxmLl9yZWluZGV4KGRmX21vbnRoLCBzZWxmLmFjdG9yX2lkcykKICAgICAgICBjb250ID0gX2FwcGx5X2xvZzFwKGRmLCBBQ1RPUl9MT0cxUCkKICAgICAgICByZXR1cm4gc2VsZi5hY3Rvcl9zY2FsZXIudHJhbnNmb3JtKAogICAgICAgICAgICBzZWxmLl9jb250X21hdHJpeChjb250LCBBQ1RPUl9DT05ULCBzZWxmLmFjdG9yX21lYW5zKQogICAgICAgICkuYXN0eXBlKCJmbG9hdDMyIikKCiAgICBkZWYgZWRnZV9mZWF0dXJlcyhzZWxmLCBkZl9yb3dzOiBwZC5EYXRhRnJhbWUpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgIiIiW0UsIGVkZ2VfZGltXSBmb3IgZWRnZSByb3dzIGFscmVhZHkgaW4gdGhlIGRlc2lyZWQgb3JkZXIgKGNhbGxlciBjb250cm9scyBvcmRlcikuCiAgICAgICAgPSBzY2FsZWQgY29udGludW91cyB8IGNsYXNzX2Rpc3RyaWJ1dGlvbls1XS4iIiIKICAgICAgICBpZiBsZW4oZGZfcm93cykgPT0gMDoKICAgICAgICAgICAgcmV0dXJuIG5wLnplcm9zKCgwLCBzZWxmLmVkZ2VfZGltKSwgZHR5cGU9ImZsb2F0MzIiKQogICAgICAgIGNvbnQgPSBfYXBwbHlfbG9nMXAoZGZfcm93cywgRURHRV9MT0cxUCkKICAgICAgICBjb250ID0gc2VsZi5lZGdlX3NjYWxlci50cmFuc2Zvcm0oc2VsZi5fY29udF9tYXRyaXgoY29udCwgRURHRV9DT05ULCBzZWxmLmVkZ2VfbWVhbnMpKS5hc3R5cGUoImZsb2F0MzIiKQogICAgICAgIGRpc3QgPSAoX3N0YWNrX2Rpc3RyaWJ1dGlvbihkZl9yb3dzWyJjbGFzc19kaXN0cmlidXRpb24iXSkKICAgICAgICAgICAgICAgIGlmICJjbGFzc19kaXN0cmlidXRpb24iIGluIGRmX3Jvd3MuY29sdW1ucwogICAgICAgICAgICAgICAgZWxzZSBucC56ZXJvcygobGVuKGRmX3Jvd3MpLCBFREdFX0RJU1RfTEVOKSwgZHR5cGU9ImZsb2F0MzIiKSkKICAgICAgICByZXR1cm4gbnAuaHN0YWNrKFtjb250LCBkaXN0XSkuYXN0eXBlKCJmbG9hdDMyIikKCiAgICAjIC0tLS0gcGVyc2lzdGVuY2UgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIHNhdmUoc2VsZiwgcGF0aDogc3RyKSAtPiBOb25lOgogICAgICAgIGpvYmxpYi5kdW1wKHNlbGYsIHBhdGgpCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGxvYWQocGF0aDogc3RyKSAtPiAiUHJlcHJvY2VzcyI6CiAgICAgICAgcmV0dXJuIGpvYmxpYi5sb2FkKHBhdGgpCg==",
"ml/losses.py": "IiIiTG9zcyBmdW5jdGlvbnMgZm9yIHRoZSA1LWNsYXNzIGVkZ2UgY2xhc3NpZmllciAocGxhbnMvMDMgwqcyLjQpLgoKQm90aCBsb3NzZXMgYXJlIG1hc2tlZCB0byBDb3VudHJ5LT5Db3VudHJ5IHRhcmdldCBwYWlycyBieSBjb25zdHJ1Y3Rpb246IHRoZSBjYWxsZXIgb25seQpwYXNzZXMgbG9naXRzL2xhYmVscyBmb3IgdGhvc2UgcGFpcnMuIENsYXNzIHdlaWdodHMgZGVmYXVsdCB0byBpbnZlcnNlLWZyZXF1ZW5jeSBvbiB0aGUKdHJhaW5pbmcgbGFiZWwgZGlzdHJpYnV0aW9uOyBTVEFUVVNfUVVPIGVuZHMgdXAgfjEuMCBhbmQgTUFURVJJQUxfQ09ORkxJQ1QgdGhlIGxhcmdlc3QuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgoKZGVmIGludmVyc2VfZnJlcXVlbmN5X3dlaWdodHMobGFiZWxfY291bnRzOiB0b3JjaC5UZW5zb3IsIG5vcm1hbGl6ZV90b19taW46IGJvb2wgPSBUcnVlKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAiIiJDbGFzcyB3ZWlnaHRzIOKInSAxIC8gZnJlcXVlbmN5LiBgbGFiZWxfY291bnRzYCBpcyBhIGxlbmd0aC1DIGxvbmcvZmxvYXQgdGVuc29yLgoKICAgIFplcm8tY291bnQgY2xhc3NlcyBhcmUgY2xhbXBlZCB0byAxIHRvIGF2b2lkIGRpdi1ieS16ZXJvLiBXaXRoIG5vcm1hbGl6ZV90b19taW4gdGhlCiAgICBzbWFsbGVzdCB3ZWlnaHQgaXMgc2NhbGVkIHRvIDEuMCAoc28gdGhlIG1ham9yaXR5IGNsYXNzIOKJiCAxLjAsIHJhcmVyIGNsYXNzZXMgPiAxKS4KICAgICIiIgogICAgY291bnRzID0gbGFiZWxfY291bnRzLmNsYW1wKG1pbj0xKS5mbG9hdCgpCiAgICB3ID0gY291bnRzLnN1bSgpIC8gY291bnRzCiAgICBpZiBub3JtYWxpemVfdG9fbWluOgogICAgICAgIHcgPSB3IC8gdy5taW4oKQogICAgcmV0dXJuIHcKCgpjbGFzcyBGb2NhbExvc3Mobm4uTW9kdWxlKToKICAgICIiIk11bHRpY2xhc3MgZm9jYWwgbG9zczogRkwgPSAtYWxwaGFfYyAqICgxIC0gcF90KV5nYW1tYSAqIGxvZyhwX3QpLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBnYW1tYTogZmxvYXQgPSAyLjAsIHdlaWdodDogdG9yY2guVGVuc29yIHwgTm9uZSA9IE5vbmUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZ2FtbWEgPSBnYW1tYQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJ3ZWlnaHQiLCB3ZWlnaHQgaWYgd2VpZ2h0IGlzIG5vdCBOb25lIGVsc2UgTm9uZSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBsb2dpdHM6IHRvcmNoLlRlbnNvciwgdGFyZ2V0OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBsb2dwID0gRi5sb2dfc29mdG1heChsb2dpdHMsIGRpbT0tMSkKICAgICAgICBsb2dwX3QgPSBsb2dwLmdhdGhlcigxLCB0YXJnZXQudW5zcXVlZXplKDEpKS5zcXVlZXplKDEpCiAgICAgICAgcF90ID0gbG9ncF90LmV4cCgpCiAgICAgICAgbG9zcyA9IC0oKDEuMCAtIHBfdCkgKiogc2VsZi5nYW1tYSkgKiBsb2dwX3QKICAgICAgICBpZiBzZWxmLndlaWdodCBpcyBub3QgTm9uZToKICAgICAgICAgICAgbG9zcyA9IGxvc3MgKiBzZWxmLndlaWdodC5nYXRoZXIoMCwgdGFyZ2V0KQogICAgICAgIHJldHVybiBsb3NzLm1lYW4oKQoKCmRlZiBtYWtlX2xvc3MobG9zc19uYW1lOiBzdHIsIGNsYXNzX3dlaWdodHM6IHRvcmNoLlRlbnNvciB8IE5vbmUsIGZvY2FsX2dhbW1hOiBmbG9hdCA9IDIuMCkgLT4gbm4uTW9kdWxlOgogICAgIiIiRmFjdG9yeTogJ3dlaWdodGVkX2NlJyAtPiBDcm9zc0VudHJvcHlMb3NzKHdlaWdodCksICdmb2NhbCcgLT4gRm9jYWxMb3NzLiIiIgogICAgaWYgbG9zc19uYW1lID09ICJmb2NhbCI6CiAgICAgICAgcmV0dXJuIEZvY2FsTG9zcyhnYW1tYT1mb2NhbF9nYW1tYSwgd2VpZ2h0PWNsYXNzX3dlaWdodHMpCiAgICBpZiBsb3NzX25hbWUgPT0gIndlaWdodGVkX2NlIjoKICAgICAgICByZXR1cm4gbm4uQ3Jvc3NFbnRyb3B5TG9zcyh3ZWlnaHQ9Y2xhc3Nfd2VpZ2h0cykKICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bmtub3duIGxvc3Mge2xvc3NfbmFtZSFyfSAoZXhwZWN0ZWQgJ3dlaWdodGVkX2NlJyBvciAnZm9jYWwnKSIpCg==",
"ml/metrics.py": "IiIiRXZhbHVhdGlvbiBtZXRyaWNzIChwbGFucy8wMyDCp0V2YWx1YXRpb24pOiBtYWNyby1GMSAocHJpbWFyeSksIHBlci1jbGFzcyBGMSwKNXg1IGNvbmZ1c2lvbiBtYXRyaXgsIGFuZCBFeHBlY3RlZCBDYWxpYnJhdGlvbiBFcnJvci4gSW1wbGVtZW50ZWQgd2l0aCB0b3JjaG1ldHJpY3Mgc28gdGhlCm51bWJlcnMgbWF0Y2ggdGhlIHN0YW5kYXJkIGRlZmluaXRpb25zOyBmYWxscyBiYWNrIHRvIG5vdGhpbmcgZXhvdGljLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2htZXRyaWNzLmNsYXNzaWZpY2F0aW9uIGltcG9ydCAoCiAgICBNdWx0aWNsYXNzQ29uZnVzaW9uTWF0cml4LAogICAgTXVsdGljbGFzc0YxU2NvcmUsCiAgICBNdWx0aWNsYXNzQ2FsaWJyYXRpb25FcnJvciwKKQoKZnJvbSAuY29uZmlnIGltcG9ydCBOVU1fQ0xBU1NFUwoKCkBkYXRhY2xhc3MKY2xhc3MgRXZhbFJlc3VsdDoKICAgIG1hY3JvX2YxOiBmbG9hdAogICAgcGVyX2NsYXNzX2YxOiBsaXN0W2Zsb2F0XQogICAgZWNlOiBmbG9hdAogICAgY29uZnVzaW9uOiBsaXN0W2xpc3RbaW50XV0KICAgIG46IGludAoKCmNsYXNzIEV2YWx1YXRvcjoKICAgICIiIkFjY3VtdWxhdGUgcHJvYmFiaWxpdGllcyArIHRhcmdldHMgYWNyb3NzIGJhdGNoZXMsIHRoZW4gYC5jb21wdXRlKClgLgoKICAgIFVzYWdlOgogICAgICAgIGV2ID0gRXZhbHVhdG9yKGRldmljZSkKICAgICAgICBldi51cGRhdGUocHJvYnMsIHRhcmdldCkgICAjIHByb2JzOiBbUCwgQ10gc29mdG1heCwgdGFyZ2V0OiBbUF0gbG9uZwogICAgICAgIHJlc3VsdCA9IGV2LmNvbXB1dGUoKQogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRldmljZTogdG9yY2guZGV2aWNlIHwgc3RyID0gImNwdSIsIG5fYmluczogaW50ID0gMTUpOgogICAgICAgIHNlbGYuZGV2aWNlID0gdG9yY2guZGV2aWNlKGRldmljZSkKICAgICAgICBzZWxmLl9tYWNybyA9IE11bHRpY2xhc3NGMVNjb3JlKG51bV9jbGFzc2VzPU5VTV9DTEFTU0VTLCBhdmVyYWdlPSJtYWNybyIpLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIHNlbGYuX3BlciA9IE11bHRpY2xhc3NGMVNjb3JlKG51bV9jbGFzc2VzPU5VTV9DTEFTU0VTLCBhdmVyYWdlPU5vbmUpLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIHNlbGYuX2NtID0gTXVsdGljbGFzc0NvbmZ1c2lvbk1hdHJpeChudW1fY2xhc3Nlcz1OVU1fQ0xBU1NFUykudG8oc2VsZi5kZXZpY2UpCiAgICAgICAgc2VsZi5fZWNlID0gTXVsdGljbGFzc0NhbGlicmF0aW9uRXJyb3IobnVtX2NsYXNzZXM9TlVNX0NMQVNTRVMsIG5fYmlucz1uX2JpbnMsIG5vcm09ImwxIikudG8oc2VsZi5kZXZpY2UpCiAgICAgICAgc2VsZi5fbiA9IDAKCiAgICBkZWYgcmVzZXQoc2VsZikgLT4gTm9uZToKICAgICAgICBmb3IgbSBpbiAoc2VsZi5fbWFjcm8sIHNlbGYuX3Blciwgc2VsZi5fY20sIHNlbGYuX2VjZSk6CiAgICAgICAgICAgIG0ucmVzZXQoKQogICAgICAgIHNlbGYuX24gPSAwCgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIHVwZGF0ZShzZWxmLCBwcm9iczogdG9yY2guVGVuc29yLCB0YXJnZXQ6IHRvcmNoLlRlbnNvcikgLT4gTm9uZToKICAgICAgICBwcm9icyA9IHByb2JzLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIHRhcmdldCA9IHRhcmdldC50byhzZWxmLmRldmljZSkKICAgICAgICBwcmVkcyA9IHByb2JzLmFyZ21heChkaW09LTEpCiAgICAgICAgc2VsZi5fbWFjcm8udXBkYXRlKHByZWRzLCB0YXJnZXQpCiAgICAgICAgc2VsZi5fcGVyLnVwZGF0ZShwcmVkcywgdGFyZ2V0KQogICAgICAgIHNlbGYuX2NtLnVwZGF0ZShwcmVkcywgdGFyZ2V0KQogICAgICAgIHNlbGYuX2VjZS51cGRhdGUocHJvYnMsIHRhcmdldCkKICAgICAgICBzZWxmLl9uICs9IGludCh0YXJnZXQubnVtZWwoKSkKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgY29tcHV0ZShzZWxmKSAtPiBFdmFsUmVzdWx0OgogICAgICAgIHJldHVybiBFdmFsUmVzdWx0KAogICAgICAgICAgICBtYWNyb19mMT1mbG9hdChzZWxmLl9tYWNyby5jb21wdXRlKCkuaXRlbSgpKSwKICAgICAgICAgICAgcGVyX2NsYXNzX2YxPVtmbG9hdCh4KSBmb3IgeCBpbiBzZWxmLl9wZXIuY29tcHV0ZSgpLnRvbGlzdCgpXSwKICAgICAgICAgICAgZWNlPWZsb2F0KHNlbGYuX2VjZS5jb21wdXRlKCkuaXRlbSgpKSwKICAgICAgICAgICAgY29uZnVzaW9uPVtbaW50KHgpIGZvciB4IGluIHJvd10gZm9yIHJvdyBpbiBzZWxmLl9jbS5jb21wdXRlKCkudG9saXN0KCldLAogICAgICAgICAgICBuPXNlbGYuX24sCiAgICAgICAgKQo=",
"ml/dataset.py": "IiIiRGF0YXNldCBidWlsZGVyOiBQYXJxdWV0IGV4cG9ydCAtPiBwZXItbW9udGggSGV0ZXJvRGF0YSB3aW5kb3dzICsgbGFiZWxlZCwgbmVnYXRpdmUtCnNhbXBsZWQgQ291bnRyeS0+Q291bnRyeSB0YXJnZXQgcGFpcnMgKHBsYW5zLzAzIMKnMS4z4oCTwqcxLjYpLgoKRGVzaWduICh0aGUgImZ1bGwtZ3JhcGgtcGVyLW1vbnRoIiByZWdpbWUsIHBsYW5zLzAzIMKnNC4xKTogZm9yIGV2ZXJ5IG1vbnRoIHRzIHdlIGJ1aWxkIG9uZQpzbWFsbCBoZXRlcm9nZW5lb3VzIGdyYXBoIChDb3VudHJ5ICsgQWN0b3Igbm9kZXMsIFNOQVBTSE9UL0JPUkRFUlMvTUVNQkVSX09GIGVkZ2VzKS4gQQp0cmFpbmluZy9ldmFsICpzYW1wbGUqIGZvciB0YXJnZXQgbW9udGggVCBpcyB0aGUgd2luZG93IG9mIFc9MTIgY29uc2VjdXRpdmUgbW9udGhseSBncmFwaHMKW1QtMTEgLi4gVF0gcGx1cyB0aGUgc2V0IG9mICh1LHYpIENvdW50cnkgcGFpcnMgd2hvc2UgbGFiZWwgaXMgdGhlIGRvbWluYW50X2NsYXNzIGF0IFQrMSwKYXVnbWVudGVkIHdpdGggSyBuZWdhdGl2ZSBTVEFUVVNfUVVPIHBhaXJzLgoKTGVha2FnZSBjb250cm9sczogdGhlIFByZXByb2Nlc3Mgc2NhbGVyIGlzIGZpdCBvbiBUUkFJTiBtb250aHMgb25seTsgYSBzYW1wbGUgYXQgVCBvbmx5IGV2ZXIKcmVhZHMgdHMgPD0gVCBmb3IgaW5wdXRzIChpdHMgb25lIGFsbG93ZWQgcGVlayBhdCB0aGUgZnV0dXJlIGlzIHRoZSBUKzEgbGFiZWwpOyB2YWwvdGVzdApuZWdhdGl2ZXMgYXJlIGZyb3plbiB3aXRoIGEgZml4ZWQgc2VlZCB3aGlsZSB0cmFpbiBuZWdhdGl2ZXMgYXJlIHJlc2FtcGxlZCBlYWNoIGVwb2NoLgoKTk9URSBvbiBzdG9yYWdlOiB0aGUgZXhwb3J0ZXIgd3JpdGVzIHRocmVlIHNpbmdsZSBQYXJxdWV0IGZpbGVzIChub2RlX3NuYXBzaG90cy5wYXJxdWV0LApzbmFwc2hvdF9lZGdlcy5wYXJxdWV0LCBzdHJ1Y3R1cmFsX2VkZ2VzLnBhcnF1ZXQpLiBPbiB0aGUgfjIzMi1ub2RlIGdyYXBoIHRoZXNlIGxvYWQgZnVsbHkKaW50byBSQU07IHRoZSBwZXItbW9udGggd2luZG93aW5nIGhhcHBlbnMgaW4tbWVtb3J5LCBzbyB0aGUgcGxhbidzIHRzLXBhcnRpdGlvbmluZyBpcyBhbgpvcHRpb25hbCBvcHRpbWl6YXRpb24gd2UgZG9uJ3QgbmVlZCBoZXJlLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoX2dlb21ldHJpYy5kYXRhIGltcG9ydCBIZXRlcm9EYXRhCgpmcm9tIC5jb25maWcgaW1wb3J0IENvbmZpZywgU1RBVFVTX1FVT19JTkRFWCwgY2xhc3NfaW5kZXgKZnJvbSAuZmVhdHVyZXMgaW1wb3J0IFByZXByb2Nlc3MKCkMsIEEgPSAiQ291bnRyeSIsICJBY3RvciIKUkVMX1NOQVAgPSAoQywgInNuYXBzaG90IiwgQykKUkVMX0JPUkRFUiA9IChDLCAiYm9yZGVycyIsIEMpClJFTF9NRU1CRVIgPSAoQywgIm1lbWJlciIsIEEpClJFTF9STUVNQkVSID0gKEEsICJyZXZfbWVtYmVyIiwgQykKCgpAZGF0YWNsYXNzCmNsYXNzIFNhbXBsZToKICAgICIiIk9uZSB0cmFpbmluZy9ldmFsIGV4YW1wbGUgZm9yIHRhcmdldCBtb250aCBULiIiIgogICAgdGFyZ2V0X3RzOiBpbnQKICAgIHBhaXJfaW5kZXg6IHRvcmNoLlRlbnNvciAgICMgW1AsIDJdIGxvbmcsIENvdW50cnkgbG9jYWwgaW5kaWNlcyAodSwgdikKICAgIHBhaXJfYXR0cjogdG9yY2guVGVuc29yICAgICMgW1AsIGVkZ2VfZGltXSBmbG9hdCwgKHUsdikgZWRnZSBmZWF0dXJlcyBhdCBtb250aCBUIChvciB6ZXJvcykKICAgIGxhYmVsczogdG9yY2guVGVuc29yICAgICAgICMgW1BdIGxvbmcsIGNsYXNzIGluZGV4IGF0IFQrMQoKCmRlZiBfdmFsaWRfYXQoc3RhcnQ6IG5wLm5kYXJyYXksIGVuZDogbnAubmRhcnJheSwgdHM6IGludCkgLT4gbnAubmRhcnJheToKICAgICIiIlBvaW50LWluLXRpbWUgdmFsaWRpdHkgcHJlZGljYXRlOiBzdGFydCA8PSB0cyBBTkQgKGVuZCBpcyBudWxsIE9SIGVuZCA+IHRzKS4iIiIKICAgIHMgPSBucC5uYW5fdG9fbnVtKHN0YXJ0LCBuYW49MC4wKQogICAgZSA9IGVuZC5hc3R5cGUoImZsb2F0NjQiKQogICAgcmV0dXJuIChzIDw9IHRzKSAmIChucC5pc25hbihlKSB8IChlID4gdHMpKQoKCmNsYXNzIEdlb3BvbGl0aWNEYXRhc2V0OgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNmZzogQ29uZmlnLCBub2RlX2RmOiBwZC5EYXRhRnJhbWUsIGVkZ2VfZGY6IHBkLkRhdGFGcmFtZSwKICAgICAgICAgICAgICAgICBzdHJ1Y3RfZGY6IHBkLkRhdGFGcmFtZSwgcHJlcHJvY2VzczogUHJlcHJvY2VzcyB8IE5vbmUgPSBOb25lKToKICAgICAgICBzZWxmLmNmZyA9IGNmZwogICAgICAgIHNlbGYubm9kZV9kZiA9IG5vZGVfZGYKICAgICAgICBzZWxmLmVkZ2VfZGYgPSBlZGdlX2RmCiAgICAgICAgc2VsZi5zdHJ1Y3RfZGYgPSBzdHJ1Y3RfZGYKCiAgICAgICAgaWYgcHJlcHJvY2VzcyBpcyBOb25lOgogICAgICAgICAgICBwcmVwcm9jZXNzID0gUHJlcHJvY2VzcyhjbGFzc19uYW1lcz1saXN0KGNmZy5jbGFzc19uYW1lcykpCiAgICAgICAgICAgIHByZXByb2Nlc3MuZml0KG5vZGVfZGYsIGVkZ2VfZGYsIGNmZy50cmFpbl9tYXhfdHMpCiAgICAgICAgc2VsZi5wcCA9IHByZXByb2Nlc3MKCiAgICAgICAgc2VsZi5jb3VudHJ5X2luZGV4ID0ge2NpZDogaSBmb3IgaSwgY2lkIGluIGVudW1lcmF0ZShzZWxmLnBwLmNvdW50cnlfaWRzKX0KICAgICAgICBzZWxmLmFjdG9yX2luZGV4ID0ge2FpZDogaSBmb3IgaSwgYWlkIGluIGVudW1lcmF0ZShzZWxmLnBwLmFjdG9yX2lkcyl9CiAgICAgICAgc2VsZi5udW1fY291bnRyeSA9IGxlbihzZWxmLnBwLmNvdW50cnlfaWRzKQogICAgICAgIHNlbGYubnVtX2FjdG9yID0gbGVuKHNlbGYucHAuYWN0b3JfaWRzKQoKICAgICAgICBzZWxmLl9idWlsZF9wZXJfdHNfdGVuc29ycygpCiAgICAgICAgc2VsZi5fYnVpbGRfbGFiZWxfaW5kZXgoKQogICAgICAgIHNlbGYuX3BhaXJfY2FjaGU6IGRpY3RbaW50LCBkaWN0W3R1cGxlW2ludCwgaW50XSwgbnAubmRhcnJheV1dID0ge30KCiAgICAjIC0tLS0gY29uc3RydWN0aW9uIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIF9idWlsZF9wZXJfdHNfdGVuc29ycyhzZWxmKSAtPiBOb25lOgogICAgICAgIGNmZyA9IHNlbGYuY2ZnCiAgICAgICAgbmQgPSBzZWxmLm5vZGVfZGYKICAgICAgICAjIGdyb3VwIG5vZGUgcm93cyBieSB0cyBmb3IgZmFzdCBwZXItbW9udGggYmxvY2tzCiAgICAgICAgbmRfY291bnRyeSA9IHt0czogZyBmb3IgdHMsIGcgaW4gbmRbbmRbIm5vZGVfdHlwZSJdID09IENdLmdyb3VwYnkoInRzIil9CiAgICAgICAgbmRfYWN0b3IgPSB7dHM6IGcgZm9yIHRzLCBnIGluIG5kW25kWyJub2RlX3R5cGUiXSA9PSBBXS5ncm91cGJ5KCJ0cyIpfQoKICAgICAgICAjIGFsbGlhbmNlIG11bHRpLWhvdCBwZXIgdHMgZnJvbSBNRU1CRVJfT0YgdmFsaWRpdHkKICAgICAgICBtZW0gPSBzZWxmLnN0cnVjdF9kZltzZWxmLnN0cnVjdF9kZlsicmVsIl0gPT0gIk1FTUJFUl9PRiJdCiAgICAgICAgbWVtX2EgPSBtZW1bImEiXS50b19udW1weSgpOyBtZW1fYiA9IG1lbVsiYiJdLnRvX251bXB5KCkKICAgICAgICBtZW1fc3RhcnQgPSBtZW1bInN0YXJ0Il0udG9fbnVtcHkoZHR5cGU9ImZsb2F0NjQiKQogICAgICAgIG1lbV9lbmQgPSBtZW1bImVuZCJdLnRvX251bXB5KGR0eXBlPSJmbG9hdDY0IikKCiAgICAgICAgYm9yZCA9IHNlbGYuc3RydWN0X2RmW3NlbGYuc3RydWN0X2RmWyJyZWwiXSA9PSAiQk9SREVSUyJdCiAgICAgICAgYl9hID0gYm9yZFsiYSJdLnRvX251bXB5KCk7IGJfYiA9IGJvcmRbImIiXS50b19udW1weSgpCiAgICAgICAgYl9zdGFydCA9IGJvcmRbInN0YXJ0Il0udG9fbnVtcHkoZHR5cGU9ImZsb2F0NjQiKQogICAgICAgIGJfZW5kID0gYm9yZFsiZW5kIl0udG9fbnVtcHkoZHR5cGU9ImZsb2F0NjQiKQoKICAgICAgICBlZF9ieV90cyA9IHt0czogZyBmb3IgdHMsIGcgaW4gc2VsZi5lZGdlX2RmLmdyb3VwYnkoInRzIil9CgogICAgICAgIHNlbGYuY291bnRyeV94OiBkaWN0W2ludCwgdG9yY2guVGVuc29yXSA9IHt9CiAgICAgICAgc2VsZi5hY3Rvcl94OiBkaWN0W2ludCwgdG9yY2guVGVuc29yXSA9IHt9CiAgICAgICAgc2VsZi5zbmFwX2luZGV4OiBkaWN0W2ludCwgdG9yY2guVGVuc29yXSA9IHt9CiAgICAgICAgc2VsZi5zbmFwX2F0dHI6IGRpY3RbaW50LCB0b3JjaC5UZW5zb3JdID0ge30KICAgICAgICBzZWxmLnNuYXBfcGFpcnM6IGRpY3RbaW50LCBsaXN0W3R1cGxlW2ludCwgaW50XV1dID0ge30KICAgICAgICBzZWxmLmJvcmRlcl9pbmRleDogZGljdFtpbnQsIHRvcmNoLlRlbnNvcl0gPSB7fQogICAgICAgIHNlbGYubWVtYmVyX2luZGV4OiBkaWN0W2ludCwgdG9yY2guVGVuc29yXSA9IHt9CgogICAgICAgIGVtcHR5MiA9IHRvcmNoLnplcm9zKCgyLCAwKSwgZHR5cGU9dG9yY2gubG9uZykKICAgICAgICBmb3IgdHMgaW4gcmFuZ2UoY2ZnLm1heF90cyArIDEpOgogICAgICAgICAgICAjIC0tLS0gbm9kZSBmZWF0dXJlcwogICAgICAgICAgICBjZGYgPSBuZF9jb3VudHJ5LmdldCh0cywgbmRfY291bnRyeS5nZXQobWF4KCh0IGZvciB0IGluIG5kX2NvdW50cnkgaWYgdCA8PSB0cyksIGRlZmF1bHQ9dHMpKSkKICAgICAgICAgICAgY2Jsb2NrID0gc2VsZi5wcC5jb3VudHJ5X2Jsb2NrKGNkZiBpZiBjZGYgaXMgbm90IE5vbmUgZWxzZSBwZC5EYXRhRnJhbWUoY29sdW1ucz1uZC5jb2x1bW5zKSkKICAgICAgICAgICAgYWxsaWFuY2UgPSBzZWxmLl9hbGxpYW5jZV9tdWx0aWhvdCh0cywgbWVtX2EsIG1lbV9iLCBtZW1fc3RhcnQsIG1lbV9lbmQpCiAgICAgICAgICAgIHNlbGYuY291bnRyeV94W3RzXSA9IHRvcmNoLmZyb21fbnVtcHkobnAuaHN0YWNrKFtjYmxvY2ssIGFsbGlhbmNlXSkuYXN0eXBlKCJmbG9hdDMyIikpCgogICAgICAgICAgICBhZGYgPSBuZF9hY3Rvci5nZXQodHMsIG5kX2FjdG9yLmdldChtYXgoKHQgZm9yIHQgaW4gbmRfYWN0b3IgaWYgdCA8PSB0cyksIGRlZmF1bHQ9dHMpKSkKICAgICAgICAgICAgYWJsb2NrID0gc2VsZi5wcC5hY3Rvcl9ibG9jayhhZGYgaWYgYWRmIGlzIG5vdCBOb25lIGVsc2UgcGQuRGF0YUZyYW1lKGNvbHVtbnM9bmQuY29sdW1ucykpCiAgICAgICAgICAgIHNlbGYuYWN0b3JfeFt0c10gPSB0b3JjaC5mcm9tX251bXB5KGFibG9jay5hc3R5cGUoImZsb2F0MzIiKSkKCiAgICAgICAgICAgICMgLS0tLSBzbmFwc2hvdCAoZXZlbnQpIGVkZ2VzIGF0IHRzCiAgICAgICAgICAgIGVkZiA9IGVkX2J5X3RzLmdldCh0cykKICAgICAgICAgICAgaWYgZWRmIGlzIG5vdCBOb25lIGFuZCBsZW4oZWRmKToKICAgICAgICAgICAgICAgIHBhaXJzLCBhdHRyID0gc2VsZi5fZWRnZXNfdG9fdGVuc29ycyhlZGYpCiAgICAgICAgICAgICAgICBzZWxmLnNuYXBfaW5kZXhbdHNdID0gcGFpcnMKICAgICAgICAgICAgICAgIHNlbGYuc25hcF9hdHRyW3RzXSA9IGF0dHIKICAgICAgICAgICAgICAgIHNlbGYuc25hcF9wYWlyc1t0c10gPSBsaXN0KG1hcCh0dXBsZSwgcGFpcnMudCgpLnRvbGlzdCgpKSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHNlbGYuc25hcF9pbmRleFt0c10gPSBlbXB0eTIuY2xvbmUoKQogICAgICAgICAgICAgICAgc2VsZi5zbmFwX2F0dHJbdHNdID0gdG9yY2guemVyb3MoKDAsIHNlbGYucHAuZWRnZV9kaW0pLCBkdHlwZT10b3JjaC5mbG9hdDMyKQogICAgICAgICAgICAgICAgc2VsZi5zbmFwX3BhaXJzW3RzXSA9IFtdCgogICAgICAgICAgICAjIC0tLS0gc3RydWN0dXJhbCBlZGdlcyB2YWxpZCBhdCB0cwogICAgICAgICAgICBzZWxmLmJvcmRlcl9pbmRleFt0c10gPSBzZWxmLl9zdHJ1Y3RfdG9faW5kZXgoYl9hLCBiX2IsIF92YWxpZF9hdChiX3N0YXJ0LCBiX2VuZCwgdHMpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNlbGYuY291bnRyeV9pbmRleCwgc2VsZi5jb3VudHJ5X2luZGV4KQogICAgICAgICAgICBzZWxmLm1lbWJlcl9pbmRleFt0c10gPSBzZWxmLl9zdHJ1Y3RfdG9faW5kZXgobWVtX2EsIG1lbV9iLCBfdmFsaWRfYXQobWVtX3N0YXJ0LCBtZW1fZW5kLCB0cyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5jb3VudHJ5X2luZGV4LCBzZWxmLmFjdG9yX2luZGV4KQoKICAgIGRlZiBfYWxsaWFuY2VfbXVsdGlob3Qoc2VsZiwgdHMsIGEsIGIsIHN0YXJ0LCBlbmQpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgb3V0ID0gbnAuemVyb3MoKHNlbGYubnVtX2NvdW50cnksIHNlbGYubnVtX2FjdG9yKSwgZHR5cGU9ImZsb2F0MzIiKQogICAgICAgIHZhbGlkID0gX3ZhbGlkX2F0KHN0YXJ0LCBlbmQsIHRzKQogICAgICAgIGZvciBpIGluIG5wLm5vbnplcm8odmFsaWQpWzBdOgogICAgICAgICAgICBjaSA9IHNlbGYuY291bnRyeV9pbmRleC5nZXQoYVtpXSk7IGFpID0gc2VsZi5hY3Rvcl9pbmRleC5nZXQoYltpXSkKICAgICAgICAgICAgaWYgY2kgaXMgbm90IE5vbmUgYW5kIGFpIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgb3V0W2NpLCBhaV0gPSAxLjAKICAgICAgICByZXR1cm4gb3V0CgogICAgZGVmIF9lZGdlc190b190ZW5zb3JzKHNlbGYsIGVkZjogcGQuRGF0YUZyYW1lKSAtPiB0dXBsZVt0b3JjaC5UZW5zb3IsIHRvcmNoLlRlbnNvcl06CiAgICAgICAgcm93cyA9IFtdCiAgICAgICAga2VlcCA9IFtdCiAgICAgICAgZm9yIHBvcywgKHMsIHQpIGluIGVudW1lcmF0ZSh6aXAoZWRmWyJzcmMiXS50b2xpc3QoKSwgZWRmWyJ0Z3QiXS50b2xpc3QoKSkpOgogICAgICAgICAgICBzaSA9IHNlbGYuY291bnRyeV9pbmRleC5nZXQocyk7IHRpID0gc2VsZi5jb3VudHJ5X2luZGV4LmdldCh0KQogICAgICAgICAgICBpZiBzaSBpcyBub3QgTm9uZSBhbmQgdGkgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICByb3dzLmFwcGVuZCgoc2ksIHRpKSk7IGtlZXAuYXBwZW5kKHBvcykKICAgICAgICBpZiBub3Qgcm93czoKICAgICAgICAgICAgcmV0dXJuIHRvcmNoLnplcm9zKCgyLCAwKSwgZHR5cGU9dG9yY2gubG9uZyksIHRvcmNoLnplcm9zKCgwLCBzZWxmLnBwLmVkZ2VfZGltKSwgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgICAgICBhdHRyID0gc2VsZi5wcC5lZGdlX2ZlYXR1cmVzKGVkZi5pbG9jW2tlZXBdKQogICAgICAgIGluZGV4ID0gdG9yY2gudGVuc29yKHJvd3MsIGR0eXBlPXRvcmNoLmxvbmcpLnQoKS5jb250aWd1b3VzKCkKICAgICAgICByZXR1cm4gaW5kZXgsIHRvcmNoLmZyb21fbnVtcHkoYXR0cikKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX3N0cnVjdF90b19pbmRleChhLCBiLCB2YWxpZCwgc3JjX2luZGV4LCBkc3RfaW5kZXgpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICByb3dzID0gW10KICAgICAgICBmb3IgaSBpbiBucC5ub256ZXJvKHZhbGlkKVswXToKICAgICAgICAgICAgc2kgPSBzcmNfaW5kZXguZ2V0KGFbaV0pOyBkaSA9IGRzdF9pbmRleC5nZXQoYltpXSkKICAgICAgICAgICAgaWYgc2kgaXMgbm90IE5vbmUgYW5kIGRpIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoKHNpLCBkaSkpCiAgICAgICAgaWYgbm90IHJvd3M6CiAgICAgICAgICAgIHJldHVybiB0b3JjaC56ZXJvcygoMiwgMCksIGR0eXBlPXRvcmNoLmxvbmcpCiAgICAgICAgcmV0dXJuIHRvcmNoLnRlbnNvcihyb3dzLCBkdHlwZT10b3JjaC5sb25nKS50KCkuY29udGlndW91cygpCgogICAgZGVmIF9idWlsZF9sYWJlbF9pbmRleChzZWxmKSAtPiBOb25lOgogICAgICAgICIiInBvc2l0aXZlc1tUXSA9IGxpc3Qgb2YgKHVfaWR4LCB2X2lkeCwgbGFiZWwpIGZyb20gc25hcHNob3QgZWRnZXMgYXQgbW9udGggVCAodGhlCiAgICAgICAgbGFiZWwgbW9udGggZm9yIHRhcmdldCBULTEpLiBXZSBpbmRleCBieSB0aGUgKmVkZ2UqIG1vbnRoIGhlcmU7IGEgdGFyZ2V0IFQgcmVhZHMgVCsxLiIiIgogICAgICAgIHNlbGYuZWRnZXNfYXQ6IGRpY3RbaW50LCBsaXN0W3R1cGxlW2ludCwgaW50LCBpbnRdXV0gPSB7fQogICAgICAgIGZvciB0cywgZyBpbiBzZWxmLmVkZ2VfZGYuZ3JvdXBieSgidHMiKToKICAgICAgICAgICAgbHN0ID0gW10KICAgICAgICAgICAgZm9yIHMsIHQsIGRjIGluIHppcChnWyJzcmMiXS50b2xpc3QoKSwgZ1sidGd0Il0udG9saXN0KCksIGdbImRvbWluYW50X2NsYXNzIl0udG9saXN0KCkpOgogICAgICAgICAgICAgICAgc2kgPSBzZWxmLmNvdW50cnlfaW5kZXguZ2V0KHMpOyB0aSA9IHNlbGYuY291bnRyeV9pbmRleC5nZXQodCkKICAgICAgICAgICAgICAgIGxhYiA9IGNsYXNzX2luZGV4KGRjKQogICAgICAgICAgICAgICAgaWYgc2kgaXMgbm90IE5vbmUgYW5kIHRpIGlzIG5vdCBOb25lIGFuZCBsYWIgPj0gMDoKICAgICAgICAgICAgICAgICAgICBsc3QuYXBwZW5kKChzaSwgdGksIGxhYikpCiAgICAgICAgICAgIHNlbGYuZWRnZXNfYXRbaW50KHRzKV0gPSBsc3QKCiAgICAjIC0tLS0gcGFpciBmZWF0dXJlIGxvb2t1cCAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIF9wYWlyX2xvb2t1cChzZWxmLCB0czogaW50KSAtPiBkaWN0W3R1cGxlW2ludCwgaW50XSwgbnAubmRhcnJheV06CiAgICAgICAgaWYgdHMgbm90IGluIHNlbGYuX3BhaXJfY2FjaGU6CiAgICAgICAgICAgIGF0dHIgPSBzZWxmLnNuYXBfYXR0clt0c10ubnVtcHkoKQogICAgICAgICAgICBzZWxmLl9wYWlyX2NhY2hlW3RzXSA9IHtwOiBhdHRyW2ldIGZvciBpLCBwIGluIGVudW1lcmF0ZShzZWxmLnNuYXBfcGFpcnNbdHNdKX0KICAgICAgICByZXR1cm4gc2VsZi5fcGFpcl9jYWNoZVt0c10KCiAgICAjIC0tLS0gcHVibGljIEFQSSAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIGJ1aWxkX3dpbmRvdyhzZWxmLCB0YXJnZXRfdHM6IGludCkgLT4gbGlzdFtIZXRlcm9EYXRhXToKICAgICAgICAiIiJUaGUgVz0xMiBtb250aGx5IGdyYXBocyBlbmRpbmcgYXQgdGFyZ2V0X3RzIChpbnB1dHMgb25seTsgbmV2ZXIgcmVhZHMgVCsxKS4iIiIKICAgICAgICBjZmcgPSBzZWxmLmNmZwogICAgICAgIG91dCA9IFtdCiAgICAgICAgZm9yIHRzIGluIHJhbmdlKHRhcmdldF90cyAtIGNmZy53aW5kb3cgKyAxLCB0YXJnZXRfdHMgKyAxKToKICAgICAgICAgICAgdHMgPSBtYXgoMCwgdHMpCiAgICAgICAgICAgIGQgPSBIZXRlcm9EYXRhKCkKICAgICAgICAgICAgZFtDXS54ID0gc2VsZi5jb3VudHJ5X3hbdHNdCiAgICAgICAgICAgIGRbQV0ueCA9IHNlbGYuYWN0b3JfeFt0c10KICAgICAgICAgICAgZFtSRUxfU05BUF0uZWRnZV9pbmRleCA9IHNlbGYuc25hcF9pbmRleFt0c10KICAgICAgICAgICAgZFtSRUxfU05BUF0uZWRnZV9hdHRyID0gc2VsZi5zbmFwX2F0dHJbdHNdCiAgICAgICAgICAgIGRbUkVMX0JPUkRFUl0uZWRnZV9pbmRleCA9IHNlbGYuYm9yZGVyX2luZGV4W3RzXQogICAgICAgICAgICBkW1JFTF9NRU1CRVJdLmVkZ2VfaW5kZXggPSBzZWxmLm1lbWJlcl9pbmRleFt0c10KICAgICAgICAgICAgZFtSRUxfUk1FTUJFUl0uZWRnZV9pbmRleCA9IHNlbGYubWVtYmVyX2luZGV4W3RzXS5mbGlwKDApCiAgICAgICAgICAgIG91dC5hcHBlbmQoZCkKICAgICAgICByZXR1cm4gb3V0CgogICAgZGVmIG1ha2Vfc2FtcGxlcyhzZWxmLCBzcGxpdDogc3RyLCBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IpIC0+IGxpc3RbU2FtcGxlXToKICAgICAgICAiIiJQb3NpdGl2ZXMgKGFsbCBlZGdlcyBhdCBUKzEpICsgSyBuZWdhdGl2ZSBTVEFUVVNfUVVPIHBhaXJzIHBlciBub24tU1EgcG9zaXRpdmUuCiAgICAgICAgUGFzcyBhIGZyZXNoIHJuZyBlYWNoIGVwb2NoIGZvciB0cmFpbjsgYSBmaXhlZC1zZWVkIHJuZyBmb3IgdmFsL3Rlc3QgKGZyb3plbikuIiIiCiAgICAgICAgY2ZnID0gc2VsZi5jZmcKICAgICAgICBzYW1wbGVzOiBsaXN0W1NhbXBsZV0gPSBbXQogICAgICAgIGVkaW0gPSBzZWxmLnBwLmVkZ2VfZGltCiAgICAgICAgZm9yIFQgaW4gY2ZnLnRhcmdldF9tb250aHMoc3BsaXQpOgogICAgICAgICAgICBsYWJlbF90cyA9IFQgKyAxCiAgICAgICAgICAgIHBvcyA9IHNlbGYuZWRnZXNfYXQuZ2V0KGxhYmVsX3RzLCBbXSkKICAgICAgICAgICAgZXhpc3RpbmcgPSB7KHUsIHYpIGZvciAodSwgdiwgXykgaW4gcG9zfQogICAgICAgICAgICBuX25vbnNxID0gc3VtKDEgZm9yIChfLCBfLCBsYWIpIGluIHBvcyBpZiBsYWIgIT0gU1RBVFVTX1FVT19JTkRFWCkKICAgICAgICAgICAgbl9uZWcgPSBjZmcua19uZWcgKiBtYXgoMSwgbl9ub25zcSkKICAgICAgICAgICAgbmVncyA9IHNlbGYuX3NhbXBsZV9uZWdhdGl2ZXMoZXhpc3RpbmcsIG5fbmVnLCBybmcpCgogICAgICAgICAgICBwYWlycyA9IFsodSwgdikgZm9yICh1LCB2LCBfKSBpbiBwb3NdICsgbmVncwogICAgICAgICAgICBsYWJlbHMgPSBbbGFiIGZvciAoXywgXywgbGFiKSBpbiBwb3NdICsgW1NUQVRVU19RVU9fSU5ERVhdICogbGVuKG5lZ3MpCiAgICAgICAgICAgIGlmIG5vdCBwYWlyczoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICBsb29rdXAgPSBzZWxmLl9wYWlyX2xvb2t1cChUKQogICAgICAgICAgICBhdHRyID0gbnAuemVyb3MoKGxlbihwYWlycyksIGVkaW0pLCBkdHlwZT0iZmxvYXQzMiIpCiAgICAgICAgICAgIGZvciBpLCAodSwgdikgaW4gZW51bWVyYXRlKHBhaXJzKToKICAgICAgICAgICAgICAgIGhpdCA9IGxvb2t1cC5nZXQoKHUsIHYpKQogICAgICAgICAgICAgICAgaWYgaGl0IGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIGF0dHJbaV0gPSBoaXQKICAgICAgICAgICAgc2FtcGxlcy5hcHBlbmQoU2FtcGxlKAogICAgICAgICAgICAgICAgdGFyZ2V0X3RzPVQsCiAgICAgICAgICAgICAgICBwYWlyX2luZGV4PXRvcmNoLnRlbnNvcihwYWlycywgZHR5cGU9dG9yY2gubG9uZyksCiAgICAgICAgICAgICAgICBwYWlyX2F0dHI9dG9yY2guZnJvbV9udW1weShhdHRyKSwKICAgICAgICAgICAgICAgIGxhYmVscz10b3JjaC50ZW5zb3IobGFiZWxzLCBkdHlwZT10b3JjaC5sb25nKSwKICAgICAgICAgICAgKSkKICAgICAgICByZXR1cm4gc2FtcGxlcwoKICAgIGRlZiBfc2FtcGxlX25lZ2F0aXZlcyhzZWxmLCBleGlzdGluZzogc2V0LCBuOiBpbnQsIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvcikgLT4gbGlzdFt0dXBsZVtpbnQsIGludF1dOgogICAgICAgIGlmIHNlbGYubnVtX2NvdW50cnkgPCAyIG9yIG4gPD0gMDoKICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgb3V0OiBsaXN0W3R1cGxlW2ludCwgaW50XV0gPSBbXQogICAgICAgIHNlZW4gPSBzZXQoZXhpc3RpbmcpCiAgICAgICAgYXR0ZW1wdHMgPSAwCiAgICAgICAgYnVkZ2V0ID0gbiAqIDIwICsgNTAKICAgICAgICB3aGlsZSBsZW4ob3V0KSA8IG4gYW5kIGF0dGVtcHRzIDwgYnVkZ2V0OgogICAgICAgICAgICBhdHRlbXB0cyArPSAxCiAgICAgICAgICAgIHUgPSBpbnQocm5nLmludGVnZXJzKHNlbGYubnVtX2NvdW50cnkpKTsgdiA9IGludChybmcuaW50ZWdlcnMoc2VsZi5udW1fY291bnRyeSkpCiAgICAgICAgICAgIGlmIHUgPT0gdiBvciAodSwgdikgaW4gc2VlbjoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHNlZW4uYWRkKCh1LCB2KSk7IG91dC5hcHBlbmQoKHUsIHYpKQogICAgICAgIHJldHVybiBvdXQKCiAgICBkZWYgY2xhc3NfY291bnRzKHNlbGYsIHNwbGl0OiBzdHIgPSAidHJhaW4iLCBzZWVkOiBpbnQgPSAwKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgY291bnRzID0gdG9yY2guemVyb3MobGVuKHNlbGYuY2ZnLmNsYXNzX25hbWVzKSwgZHR5cGU9dG9yY2gubG9uZykKICAgICAgICBmb3IgcyBpbiBzZWxmLm1ha2Vfc2FtcGxlcyhzcGxpdCwgbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpKToKICAgICAgICAgICAgY291bnRzICs9IHRvcmNoLmJpbmNvdW50KHMubGFiZWxzLCBtaW5sZW5ndGg9bGVuKHNlbGYuY2ZnLmNsYXNzX25hbWVzKSkKICAgICAgICByZXR1cm4gY291bnRzCgogICAgIyAtLS0tIGxvYWRpbmcgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGZyb21fcGFycXVldChjbHMsIGNmZzogQ29uZmlnLCBwcmVwcm9jZXNzOiBQcmVwcm9jZXNzIHwgTm9uZSA9IE5vbmUpIC0+ICJHZW9wb2xpdGljRGF0YXNldCI6CiAgICAgICAgZCA9IGNmZy5kYXRhX2RpcgogICAgICAgIG5vZGVfZGYgPSBwZC5yZWFkX3BhcnF1ZXQob3MucGF0aC5qb2luKGQsICJub2RlX3NuYXBzaG90cy5wYXJxdWV0IikpCiAgICAgICAgZWRnZV9kZiA9IHBkLnJlYWRfcGFycXVldChvcy5wYXRoLmpvaW4oZCwgInNuYXBzaG90X2VkZ2VzLnBhcnF1ZXQiKSkKICAgICAgICBzdHJ1Y3RfZGYgPSBwZC5yZWFkX3BhcnF1ZXQob3MucGF0aC5qb2luKGQsICJzdHJ1Y3R1cmFsX2VkZ2VzLnBhcnF1ZXQiKSkKICAgICAgICBmb3IgZGYgaW4gKG5vZGVfZGYsIGVkZ2VfZGYpOgogICAgICAgICAgICBkZlsidHMiXSA9IGRmWyJ0cyJdLmFzdHlwZShpbnQpCiAgICAgICAgcmV0dXJuIGNscyhjZmcsIG5vZGVfZGYsIGVkZ2VfZGYsIHN0cnVjdF9kZiwgcHJlcHJvY2VzcykK",
"ml/model.py": "IiIiVGhlIG1vZGVsIChwbGFucy8wMyDCpzIpOiBhIGhldGVyb2dlbmVvdXMsIGVkZ2UtYXdhcmUgc3BhdGlvLXRlbXBvcmFsIEdOTi4KClBlciBtb250aDogYSBzaGFyZWQgMi1ob3AgaGV0ZXJvZ2VuZW91cyBlbmNvZGVyIChIZXRlcm9Db252IHdyYXBwaW5nIFRyYW5zZm9ybWVyQ29udiBvbiB0aGUKZWRnZS1mZWF0dXJlLXJpY2ggU05BUFNIT1QgcmVsYXRpb24gKyBHQVR2MkNvbnYgb24gdGhlIHN0cnVjdHVyYWwgcmVsYXRpb25zKSBwcm9kdWNlcyBhCkNvdW50cnkgZW1iZWRkaW5nLiBUaGUgVz0xMiBtb250aGx5IENvdW50cnkgZW1iZWRkaW5ncyBmb3JtIGEgcGVyLW5vZGUgc2VxdWVuY2UgZmVkIHRvIGEgR1JVLAp0aGVuIGF0dGVudGlvbi1wb29sZWQgb3ZlciB0aW1lLiBBIGRpcmVjdGlvbi1hd2FyZSBjb25jYXQrTUxQIGRlY29kZXIgbWFwcyAoaF91LCBoX3YsIGVkZ2UpCnRvIDUgY2xhc3MgbG9naXRzLiBJZGVudGl0eSBlbWJlZGRpbmdzIGFyZSBjb25jYXRlbmF0ZWQgYXQgaW5wdXQsIHBlciBub2RlIHR5cGUuCgpGb3J3YXJkIGNvbnN1bWVzIG9uZSB3aW5kb3cgPSBsaXN0W0hldGVyb0RhdGFdIChsZW5ndGggVyk7IHRoZSBncmFwaHMgYXJlIHRpbnkgKH4yMzIgbm9kZXMpCnNvIHdlIGVuY29kZSB0aGVtIGluIGEgUHl0aG9uIGxvb3AgYW5kIHN0YWNrIOKAlCB0aGUgb25seSAiY3VzdG9tIGdsdWUiIHRoZSBwbGFuIGNhbGxzIG91dC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKZnJvbSB0b3JjaF9nZW9tZXRyaWMubm4gaW1wb3J0IEdBVHYyQ29udiwgSGV0ZXJvQ29udiwgVHJhbnNmb3JtZXJDb252Cgpmcm9tIC5jb25maWcgaW1wb3J0IENvbmZpZywgTlVNX0NMQVNTRVMKZnJvbSAuZGF0YXNldCBpbXBvcnQgQywgQSwgUkVMX1NOQVAsIFJFTF9CT1JERVIsIFJFTF9NRU1CRVIsIFJFTF9STUVNQkVSCgoKY2xhc3MgU3BhdGlvVGVtcG9yYWxFZGdlQ2xhc3NpZmllcihubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG51bV9jb3VudHJ5OiBpbnQsIG51bV9hY3RvcjogaW50LCBjb3VudHJ5X2luOiBpbnQsIGFjdG9yX2luOiBpbnQsCiAgICAgICAgICAgICAgICAgZWRnZV9kaW06IGludCwgY2ZnOiBDb25maWcpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGQsIGggPSBjZmcuaGlkZGVuLCBjZmcuaGVhZHMKICAgICAgICBhc3NlcnQgZCAlIGggPT0gMCwgImhpZGRlbiBtdXN0IGJlIGRpdmlzaWJsZSBieSBoZWFkcyIKICAgICAgICBzZWxmLmNmZyA9IGNmZwogICAgICAgIHNlbGYubnVtX2NvdW50cnkgPSBudW1fY291bnRyeQogICAgICAgIHNlbGYubnVtX2FjdG9yID0gbnVtX2FjdG9yCgogICAgICAgIHNlbGYuZW1iX2NvdW50cnkgPSBubi5FbWJlZGRpbmcobnVtX2NvdW50cnksIGNmZy5pZF9kaW0pCiAgICAgICAgc2VsZi5lbWJfYWN0b3IgPSBubi5FbWJlZGRpbmcobnVtX2FjdG9yLCBjZmcuaWRfZGltKQogICAgICAgIHNlbGYuaW5fY291bnRyeSA9IG5uLkxpbmVhcihjb3VudHJ5X2luICsgY2ZnLmlkX2RpbSwgZCkKICAgICAgICBzZWxmLmluX2FjdG9yID0gbm4uTGluZWFyKGFjdG9yX2luICsgY2ZnLmlkX2RpbSwgZCkKCiAgICAgICAgc2VsZi5jb252cyA9IG5uLk1vZHVsZUxpc3QoKQogICAgICAgIHNlbGYubm9ybV9jID0gbm4uTW9kdWxlTGlzdCgpCiAgICAgICAgc2VsZi5ub3JtX2EgPSBubi5Nb2R1bGVMaXN0KCkKICAgICAgICBmb3IgXyBpbiByYW5nZShjZmcuaG9wcyk6CiAgICAgICAgICAgIGNvbnYgPSBIZXRlcm9Db252KHsKICAgICAgICAgICAgICAgIFJFTF9TTkFQOiBUcmFuc2Zvcm1lckNvbnYoZCwgZCAvLyBoLCBoZWFkcz1oLCBlZGdlX2RpbT1lZGdlX2RpbSwgYmV0YT1UcnVlLCBkcm9wb3V0PWNmZy5kcm9wb3V0KSwKICAgICAgICAgICAgICAgIFJFTF9CT1JERVI6IEdBVHYyQ29udihkLCBkIC8vIGgsIGhlYWRzPWgsIGFkZF9zZWxmX2xvb3BzPVRydWUsIGRyb3BvdXQ9Y2ZnLmRyb3BvdXQpLAogICAgICAgICAgICAgICAgUkVMX01FTUJFUjogR0FUdjJDb252KChkLCBkKSwgZCAvLyBoLCBoZWFkcz1oLCBhZGRfc2VsZl9sb29wcz1GYWxzZSwgZHJvcG91dD1jZmcuZHJvcG91dCksCiAgICAgICAgICAgICAgICBSRUxfUk1FTUJFUjogR0FUdjJDb252KChkLCBkKSwgZCAvLyBoLCBoZWFkcz1oLCBhZGRfc2VsZl9sb29wcz1GYWxzZSwgZHJvcG91dD1jZmcuZHJvcG91dCksCiAgICAgICAgICAgIH0sIGFnZ3I9InN1bSIpCiAgICAgICAgICAgIHNlbGYuY29udnMuYXBwZW5kKGNvbnYpCiAgICAgICAgICAgIHNlbGYubm9ybV9jLmFwcGVuZChubi5MYXllck5vcm0oZCkpCiAgICAgICAgICAgIHNlbGYubm9ybV9hLmFwcGVuZChubi5MYXllck5vcm0oZCkpCgogICAgICAgIHNlbGYuZ3J1ID0gbm4uR1JVKGQsIGQsIG51bV9sYXllcnM9Y2ZnLmdydV9sYXllcnMsIGJhdGNoX2ZpcnN0PVRydWUpCiAgICAgICAgc2VsZi50ZW1wb3JhbF9hdHRuID0gbm4uTGluZWFyKGQsIDEpCiAgICAgICAgc2VsZi5kcm9wb3V0ID0gbm4uRHJvcG91dChjZmcuZHJvcG91dCkKICAgICAgICBzZWxmLmRlY29kZXIgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoMyAqIGQgKyBlZGdlX2RpbSwgZCksIG5uLlJlTFUoKSwgbm4uRHJvcG91dChjZmcuZHJvcG91dCksIG5uLkxpbmVhcihkLCBOVU1fQ0xBU1NFUyksCiAgICAgICAgKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJfY19pZHMiLCB0b3JjaC5hcmFuZ2UobnVtX2NvdW50cnkpLCBwZXJzaXN0ZW50PUZhbHNlKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJfYV9pZHMiLCB0b3JjaC5hcmFuZ2UobnVtX2FjdG9yKSwgcGVyc2lzdGVudD1GYWxzZSkKCiAgICAjIC0tIG9uZSBtb250aGx5IGdyYXBoIC0+IENvdW50cnkgZW1iZWRkaW5ncyBbTmMsIGRdIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIGVuY29kZV9zbmFwc2hvdChzZWxmLCBkYXRhKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgeGMgPSB0b3JjaC5jYXQoW2RhdGFbQ10ueCwgc2VsZi5lbWJfY291bnRyeShzZWxmLl9jX2lkcyldLCBkaW09LTEpCiAgICAgICAgeGEgPSB0b3JjaC5jYXQoW2RhdGFbQV0ueCwgc2VsZi5lbWJfYWN0b3Ioc2VsZi5fYV9pZHMpXSwgZGltPS0xKQogICAgICAgIHhfZGljdCA9IHtDOiBGLnJlbHUoc2VsZi5pbl9jb3VudHJ5KHhjKSksIEE6IEYucmVsdShzZWxmLmluX2FjdG9yKHhhKSl9CgogICAgICAgIGVkZ2VfaW5kZXhfZGljdCA9IHsKICAgICAgICAgICAgUkVMX1NOQVA6IGRhdGFbUkVMX1NOQVBdLmVkZ2VfaW5kZXgsCiAgICAgICAgICAgIFJFTF9CT1JERVI6IGRhdGFbUkVMX0JPUkRFUl0uZWRnZV9pbmRleCwKICAgICAgICAgICAgUkVMX01FTUJFUjogZGF0YVtSRUxfTUVNQkVSXS5lZGdlX2luZGV4LAogICAgICAgICAgICBSRUxfUk1FTUJFUjogZGF0YVtSRUxfUk1FTUJFUl0uZWRnZV9pbmRleCwKICAgICAgICB9CiAgICAgICAgZWRnZV9hdHRyX2RpY3QgPSB7UkVMX1NOQVA6IGRhdGFbUkVMX1NOQVBdLmVkZ2VfYXR0cn0KCiAgICAgICAgZm9yIGNvbnYsIG5jLCBuYSBpbiB6aXAoc2VsZi5jb252cywgc2VsZi5ub3JtX2MsIHNlbGYubm9ybV9hKToKICAgICAgICAgICAgb3V0ID0gY29udih4X2RpY3QsIGVkZ2VfaW5kZXhfZGljdCwgZWRnZV9hdHRyX2RpY3Q9ZWRnZV9hdHRyX2RpY3QpCiAgICAgICAgICAgICMgLmdldCBmYWxsYmFjazogYSBtb250aCBtYXkgaGF2ZSBubyBlZGdlcyBvZiBzb21lIHJlbGF0aW9uIC0+IGtlZXAgcHJpb3Igc3RhdGUuCiAgICAgICAgICAgIG9jID0gb3V0LmdldChDLCB4X2RpY3RbQ10pCiAgICAgICAgICAgIG9hID0gb3V0LmdldChBLCB4X2RpY3RbQV0pCiAgICAgICAgICAgIGhjID0gc2VsZi5kcm9wb3V0KEYuZWx1KG5jKG9jKSkpICsgeF9kaWN0W0NdICAgIyByZXNpZHVhbCAobWl0aWdhdGVzIG92ZXJzbW9vdGhpbmcpCiAgICAgICAgICAgIGhhID0gc2VsZi5kcm9wb3V0KEYuZWx1KG5hKG9hKSkpICsgeF9kaWN0W0FdCiAgICAgICAgICAgIHhfZGljdCA9IHtDOiBoYywgQTogaGF9CiAgICAgICAgcmV0dXJuIHhfZGljdFtDXQoKICAgICMgLS0gZnVsbCB3aW5kb3cgLT4gNS1jbGFzcyBsb2dpdHMgcGVyIHRhcmdldCBwYWlyIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBkZWYgZm9yd2FyZChzZWxmLCB3aW5kb3c6IGxpc3QsIHBhaXJfaW5kZXg6IHRvcmNoLlRlbnNvciwgcGFpcl9hdHRyOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBzZXEgPSBbc2VsZi5lbmNvZGVfc25hcHNob3QoZCkgZm9yIGQgaW4gd2luZG93XSAgICAgICMgVyB4IFtOYywgZF0KICAgICAgICBIID0gdG9yY2guc3RhY2soc2VxLCBkaW09MSkgICAgICAgICAgICAgICAgICAgICAgICAgICMgW05jLCBXLCBkXQogICAgICAgIGdydV9vdXQsIF8gPSBzZWxmLmdydShIKSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBbTmMsIFcsIGRdCiAgICAgICAgYXR0biA9IHRvcmNoLnNvZnRtYXgoc2VsZi50ZW1wb3JhbF9hdHRuKGdydV9vdXQpLCBkaW09MSkgICAjIFtOYywgVywgMV0KICAgICAgICBoID0gKGdydV9vdXQgKiBhdHRuKS5zdW0oZGltPTEpICAgICAgICAgICAgICAgICAgICAgICMgW05jLCBkXQoKICAgICAgICB1ID0gaFtwYWlyX2luZGV4WzosIDBdXQogICAgICAgIHYgPSBoW3BhaXJfaW5kZXhbOiwgMV1dCiAgICAgICAgZmVhdCA9IHRvcmNoLmNhdChbdSwgdiwgdSAqIHYsIHBhaXJfYXR0cl0sIGRpbT0tMSkKICAgICAgICByZXR1cm4gc2VsZi5kZWNvZGVyKGZlYXQpICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgW1AsIDVdIGxvZ2l0cwoKICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGZyb21fZGF0YXNldChjbHMsIGRzLCBjZmc6IENvbmZpZykgLT4gIlNwYXRpb1RlbXBvcmFsRWRnZUNsYXNzaWZpZXIiOgogICAgICAgIHJldHVybiBjbHMoCiAgICAgICAgICAgIG51bV9jb3VudHJ5PWRzLm51bV9jb3VudHJ5LCBudW1fYWN0b3I9ZHMubnVtX2FjdG9yLAogICAgICAgICAgICBjb3VudHJ5X2luPWRzLnBwLmNvdW50cnlfZmVhdF9kaW0sIGFjdG9yX2luPWRzLnBwLmFjdG9yX2ZlYXRfZGltLAogICAgICAgICAgICBlZGdlX2RpbT1kcy5wcC5lZGdlX2RpbSwgY2ZnPWNmZywKICAgICAgICApCg==",
"ml/calibrate.py": "IiIiUHJvYmFiaWxpdHkgY2FsaWJyYXRpb24gKHBsYW5zLzAzIMKnMi41KS4gVGVtcGVyYXR1cmUgc2NhbGluZzogYSBzaW5nbGUgc2NhbGFyIFQgbGVhcm5lZApvbiB0aGUgdmFsaWRhdGlvbiBzZXQgdGhhdCBkaXZpZGVzIHRoZSBsb2dpdHMgYmVmb3JlIHNvZnRtYXgsIG1pbmltaXppbmcgTkxMLiBDaGVhcCwga2VlcHMKdGhlIGFyZ21heCB1bmNoYW5nZWQsIGFuZCByZWxpYWJseSBsb3dlcnMgRUNFLiBTZXJpYWxpemVkIGluc2lkZSBjYWxpYnJhdG9yLnBrbC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgam9ibGliCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKCmNsYXNzIFRlbXBlcmF0dXJlU2NhbGVyKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMS4wKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmxvZ190ID0gbm4uUGFyYW1ldGVyKHRvcmNoLnRlbnNvcihmbG9hdCh0b3JjaC5sb2codG9yY2gudGVuc29yKHRlbXBlcmF0dXJlKSkpKSkKCiAgICBAcHJvcGVydHkKICAgIGRlZiB0ZW1wZXJhdHVyZShzZWxmKSAtPiBmbG9hdDoKICAgICAgICByZXR1cm4gZmxvYXQoc2VsZi5sb2dfdC5leHAoKS5pdGVtKCkpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgbG9naXRzOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICByZXR1cm4gbG9naXRzIC8gc2VsZi5sb2dfdC5leHAoKQoKICAgIGRlZiBwcm9icyhzZWxmLCBsb2dpdHM6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHJldHVybiBGLnNvZnRtYXgoc2VsZi5mb3J3YXJkKGxvZ2l0cyksIGRpbT0tMSkKCiAgICBkZWYgZml0KHNlbGYsIGxvZ2l0czogdG9yY2guVGVuc29yLCB0YXJnZXQ6IHRvcmNoLlRlbnNvciwgbWF4X2l0ZXI6IGludCA9IDEwMCkgLT4gIlRlbXBlcmF0dXJlU2NhbGVyIjoKICAgICAgICAiIiJMZWFybiBUIGJ5IG1pbmltaXppbmcgTkxMIG9uIGhlbGQtb3V0ICh2YWwpIGxvZ2l0cyB3aXRoIExCRkdTLiIiIgogICAgICAgIGxvZ2l0cyA9IGxvZ2l0cy5kZXRhY2goKQogICAgICAgIHRhcmdldCA9IHRhcmdldC5kZXRhY2goKQogICAgICAgIG9wdCA9IHRvcmNoLm9wdGltLkxCRkdTKFtzZWxmLmxvZ190XSwgbHI9MC4wNSwgbWF4X2l0ZXI9bWF4X2l0ZXIpCiAgICAgICAgbmxsID0gbm4uQ3Jvc3NFbnRyb3B5TG9zcygpCgogICAgICAgIGRlZiBjbG9zdXJlKCk6CiAgICAgICAgICAgIG9wdC56ZXJvX2dyYWQoKQogICAgICAgICAgICBsb3NzID0gbmxsKHNlbGYuZm9yd2FyZChsb2dpdHMpLCB0YXJnZXQpCiAgICAgICAgICAgIGxvc3MuYmFja3dhcmQoKQogICAgICAgICAgICByZXR1cm4gbG9zcwoKICAgICAgICBvcHQuc3RlcChjbG9zdXJlKQogICAgICAgIHJldHVybiBzZWxmCgogICAgZGVmIHNhdmUoc2VsZiwgcGF0aDogc3RyKSAtPiBOb25lOgogICAgICAgIGpvYmxpYi5kdW1wKHsidGVtcGVyYXR1cmUiOiBzZWxmLnRlbXBlcmF0dXJlfSwgcGF0aCkKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgbG9hZChwYXRoOiBzdHIpIC0+ICJUZW1wZXJhdHVyZVNjYWxlciI6CiAgICAgICAgcmV0dXJuIFRlbXBlcmF0dXJlU2NhbGVyKGpvYmxpYi5sb2FkKHBhdGgpWyJ0ZW1wZXJhdHVyZSJdKQo=",
"ml/explain.py": "IiIiRXhwbGFpbmFiaWxpdHkgKHBsYW5zLzAzIMKnMyk6IHdoeSB0aGUgbW9kZWwgcHJlZGljdGVkIGEgY2xhc3MgZm9yIGEgQ291bnRyeSBwYWlyICh1LHYpLgoKVHdvIGNvbXBsZW1lbnRhcnkgbWV0aG9kcywgaW1wbGVtZW50ZWQgdG8gd29yayB3aXRoIHRoaXMgbW9kZWwncyBjdXN0b20gdGVtcG9yYWwgZm9yd2FyZApzaWduYXR1cmUgKGEgd2luZG93ID0gbGlzdFtIZXRlcm9EYXRhXSkgcmF0aGVyIHRoYW4gdGhlIHN0YW5kYXJkICh4LCBlZGdlX2luZGV4KSBzaWduYXR1cmU6CgotIGBpbnRlZ3JhdGVkX2dyYWRpZW50c2Ag4oCUIGZlYXR1cmUtbGV2ZWwuIEludGVncmF0ZXMgdGhlIGdyYWRpZW50IG9mIHRoZSBwcmVkaWN0ZWQtY2xhc3MKICBwcm9iYWJpbGl0eSBmcm9tIGEgemVybyBiYXNlbGluZSB0byB0aGUgcmVhbCBpbnB1dCwgZm9yIChhKSB0aGUgKHUsdikgZWRnZSBmZWF0dXJlIHZlY3RvcgogIGFuZCAoYikgdSdzIGFuZCB2J3Mgbm9kZSBmZWF0dXJlcyBpbiB0aGUgbGFzdCB3aW5kb3cgbW9udGguIFNhdGlzZmllcyB0aGUgY29tcGxldGVuZXNzCiAgYXhpb20gKHN1bSBvZiBhdHRyaWJ1dGlvbnMg4omIIEYoeCkg4oiSIEYoYmFzZWxpbmUpKTsgYGNvbXBsZXRlbmVzc19nYXBgIHJlcG9ydHMgdGhlIHJlc2lkdWFsLgotIGBnbm5fZXhwbGFpbmVyX2VkZ2VfbWFza2Ag4oCUIHN0cnVjdHVyZS1sZXZlbC4gTGVhcm5zIGEgc29mdCBtYXNrIG92ZXIgdGhlIGxhc3QgbW9udGgncwogIFNOQVBTSE9UIGVkZ2VzIChHTk5FeHBsYWluZXItc3R5bGUpIHRoYXQgcHJlc2VydmVzIHRoZSBwcmVkaWN0ZWQgY2xhc3Mgd2l0aCBhIHNwYXJzaXR5CiAgcGVuYWx0eTsgcmV0dXJucyBwZXItZWRnZSBpbXBvcnRhbmNlcyBpbiBbMCwxXS4KCkJvdGggb3BlcmF0ZSBvbiBhIHNpbmdsZSBwYWlyIGF0IGEgdGltZSAodGhlIGluZmVyZW5jZSBwYXRoKSwgYXMgdGhlIHBsYW4gcmVxdWlyZXMuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKZnJvbSAuZGF0YXNldCBpbXBvcnQgQywgUkVMX1NOQVAKCgpkZWYgX2Nsb25lX3dpbmRvdyh3aW5kb3c6IGxpc3QpIC0+IGxpc3Q6CiAgICByZXR1cm4gW2QuY2xvbmUoKSBmb3IgZCBpbiB3aW5kb3ddCgoKZGVmIF9wcm9iX2Zvcl9wYWlyKG1vZGVsLCB3aW5kb3csIHU6IGludCwgdjogaW50LCBjbHM6IGludCkgLT4gdG9yY2guVGVuc29yOgogICAgcGFpciA9IHRvcmNoLnRlbnNvcihbW3UsIHZdXSwgZHR5cGU9dG9yY2gubG9uZywgZGV2aWNlPXdpbmRvd1swXVtDXS54LmRldmljZSkKICAgIGF0dHIgPSB3aW5kb3dbLTFdW1JFTF9TTkFQXS5lZGdlX2F0dHIgICMgcGxhY2Vob2xkZXI7IGNhbGxlciBzdXBwbGllcyBwYWlyX2F0dHIgc2VwYXJhdGVseQogICAgbG9naXRzID0gbW9kZWwod2luZG93LCBwYWlyLCBhdHRyLm5ld196ZXJvcygoMSwgYXR0ci5zaGFwZVsxXSkpKQogICAgcmV0dXJuIEYuc29mdG1heChsb2dpdHMsIGRpbT0tMSlbMCwgY2xzXQoKCkBkYXRhY2xhc3MKY2xhc3MgSUdSZXN1bHQ6CiAgICBlZGdlX2F0dHJpYnV0aW9uOiBsaXN0W2Zsb2F0XSAgICAgICMgcGVyIGRpbSBvZiB0aGUgKHUsdikgZWRnZSBmZWF0dXJlIHZlY3RvcgogICAgdV9ub2RlX2F0dHJpYnV0aW9uOiBsaXN0W2Zsb2F0XSAgICAjIHBlciBkaW0gb2YgdSdzIGxhc3QtbW9udGggbm9kZSBmZWF0dXJlcwogICAgdl9ub2RlX2F0dHJpYnV0aW9uOiBsaXN0W2Zsb2F0XQogICAgY29tcGxldGVuZXNzX2dhcDogZmxvYXQgICAgICAgICAgICAjIHxzdW0oYXR0cikgLSAoRih4KSAtIEYoYmFzZWxpbmUpKXwKICAgIHRhcmdldF9jbGFzczogaW50CgoKQHRvcmNoLmVuYWJsZV9ncmFkKCkKZGVmIGludGVncmF0ZWRfZ3JhZGllbnRzKG1vZGVsLCB3aW5kb3c6IGxpc3QsIHU6IGludCwgdjogaW50LCBwYWlyX2F0dHI6IHRvcmNoLlRlbnNvciwKICAgICAgICAgICAgICAgICAgICAgICAgIHRhcmdldF9jbGFzczogaW50IHwgTm9uZSA9IE5vbmUsIHN0ZXBzOiBpbnQgPSA1MCkgLT4gSUdSZXN1bHQ6CiAgICBtb2RlbC5ldmFsKCkKICAgIGRldmljZSA9IHdpbmRvd1swXVtDXS54LmRldmljZQogICAgcGFpciA9IHRvcmNoLnRlbnNvcihbW3UsIHZdXSwgZHR5cGU9dG9yY2gubG9uZywgZGV2aWNlPWRldmljZSkKCiAgICBiYXNlX2F0dHIgPSB0b3JjaC56ZXJvc19saWtlKHBhaXJfYXR0cikKICAgIGxhc3RfeCA9IHdpbmRvd1stMV1bQ10ueAogICAgYmFzZV94ID0gdG9yY2guemVyb3NfbGlrZShsYXN0X3gpCgogICAgaWYgdGFyZ2V0X2NsYXNzIGlzIE5vbmU6CiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIHRhcmdldF9jbGFzcyA9IGludChtb2RlbCh3aW5kb3csIHBhaXIsIHBhaXJfYXR0cikuYXJnbWF4KGRpbT0tMSkuaXRlbSgpKQoKICAgIGdyYWRfYXR0ciA9IHRvcmNoLnplcm9zX2xpa2UocGFpcl9hdHRyKQogICAgZ3JhZF94ID0gdG9yY2guemVyb3NfbGlrZShsYXN0X3gpCiAgICBmb3IgayBpbiByYW5nZSgxLCBzdGVwcyArIDEpOgogICAgICAgIGFscGhhID0gayAvIHN0ZXBzCiAgICAgICAgYSA9IChiYXNlX2F0dHIgKyBhbHBoYSAqIChwYWlyX2F0dHIgLSBiYXNlX2F0dHIpKS5kZXRhY2goKS5yZXF1aXJlc19ncmFkXyhUcnVlKQogICAgICAgIHcgPSBfY2xvbmVfd2luZG93KHdpbmRvdykKICAgICAgICB4ID0gKGJhc2VfeCArIGFscGhhICogKGxhc3RfeCAtIGJhc2VfeCkpLmRldGFjaCgpLnJlcXVpcmVzX2dyYWRfKFRydWUpCiAgICAgICAgd1stMV1bQ10ueCA9IHgKICAgICAgICBwID0gRi5zb2Z0bWF4KG1vZGVsKHcsIHBhaXIsIGEpLCBkaW09LTEpWzAsIHRhcmdldF9jbGFzc10KICAgICAgICBnYSwgZ3ggPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKHAsIFthLCB4XSwgcmV0YWluX2dyYXBoPUZhbHNlKQogICAgICAgIGdyYWRfYXR0ciArPSBnYS5kZXRhY2goKQogICAgICAgIGdyYWRfeCArPSBneC5kZXRhY2goKQoKICAgIGlnX2F0dHIgPSAoKHBhaXJfYXR0ciAtIGJhc2VfYXR0cikgKiBncmFkX2F0dHIgLyBzdGVwcykuc3F1ZWV6ZSgwKQogICAgaWdfeCA9IChsYXN0X3ggLSBiYXNlX3gpICogZ3JhZF94IC8gc3RlcHMKCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBmX3ggPSBmbG9hdChGLnNvZnRtYXgobW9kZWwod2luZG93LCBwYWlyLCBwYWlyX2F0dHIpLCBkaW09LTEpWzAsIHRhcmdldF9jbGFzc10pCiAgICAgICAgdzAgPSBfY2xvbmVfd2luZG93KHdpbmRvdykKICAgICAgICB3MFstMV1bQ10ueCA9IGJhc2VfeAogICAgICAgIGZfYmFzZSA9IGZsb2F0KEYuc29mdG1heChtb2RlbCh3MCwgcGFpciwgYmFzZV9hdHRyKSwgZGltPS0xKVswLCB0YXJnZXRfY2xhc3NdKQogICAgIyBDb21wbGV0ZW5lc3MgaXMgb3ZlciBBTEwgcGVydHVyYmVkIGlucHV0czogdGhlICh1LHYpIGVkZ2UgdmVjdG9yICsgRVZFUlkgbGFzdC1tb250aAogICAgIyBub2RlIGZlYXR1cmUgaW50ZXJwb2xhdGVkIGZyb20gdGhlIHplcm8gYmFzZWxpbmUgKG5vdCBqdXN0IHUvdiwgd2hpY2ggYXJlIHJlcG9ydGVkIG9ubHkKICAgICMgZm9yIGludGVycHJldGFiaWxpdHkpLiBTdW1taW5nIGp1c3QgdS92IHdvdWxkIGxlYXZlIGEgbGFyZ2Ugc3B1cmlvdXMgcmVzaWR1YWwuCiAgICB0b3RhbCA9IGZsb2F0KGlnX2F0dHIuc3VtKCkgKyBpZ194LnN1bSgpKQogICAgcmV0dXJuIElHUmVzdWx0KAogICAgICAgIGVkZ2VfYXR0cmlidXRpb249aWdfYXR0ci50b2xpc3QoKSwKICAgICAgICB1X25vZGVfYXR0cmlidXRpb249aWdfeFt1XS50b2xpc3QoKSwKICAgICAgICB2X25vZGVfYXR0cmlidXRpb249aWdfeFt2XS50b2xpc3QoKSwKICAgICAgICBjb21wbGV0ZW5lc3NfZ2FwPWFicyh0b3RhbCAtIChmX3ggLSBmX2Jhc2UpKSwKICAgICAgICB0YXJnZXRfY2xhc3M9dGFyZ2V0X2NsYXNzLAogICAgKQoKCkB0b3JjaC5lbmFibGVfZ3JhZCgpCmRlZiBnbm5fZXhwbGFpbmVyX2VkZ2VfbWFzayhtb2RlbCwgd2luZG93OiBsaXN0LCB1OiBpbnQsIHY6IGludCwgcGFpcl9hdHRyOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB0YXJnZXRfY2xhc3M6IGludCB8IE5vbmUgPSBOb25lLCBlcG9jaHM6IGludCA9IDIwMCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxyOiBmbG9hdCA9IDAuMDEsIGNvZWZmX3NpemU6IGZsb2F0ID0gMWUtMykgLT4gbGlzdFtmbG9hdF06CiAgICAiIiJMZWFybiBhIHNpZ21vaWQgbWFzayBvdmVyIHRoZSBsYXN0LW1vbnRoIFNOQVBTSE9UIGVkZ2VzIHRoYXQgcHJlc2VydmVzIHRoZSBwcmVkaWN0ZWQKICAgIGNsYXNzIChHTk5FeHBsYWluZXItc3R5bGUpLiBSZXR1cm5zIHBlci1lZGdlIGltcG9ydGFuY2UgaW4gWzAsMV0uIiIiCiAgICBtb2RlbC5ldmFsKCkKICAgIGRldmljZSA9IHdpbmRvd1swXVtDXS54LmRldmljZQogICAgcGFpciA9IHRvcmNoLnRlbnNvcihbW3UsIHZdXSwgZHR5cGU9dG9yY2gubG9uZywgZGV2aWNlPWRldmljZSkKICAgIGJhc2VfYXR0ciA9IHdpbmRvd1stMV1bUkVMX1NOQVBdLmVkZ2VfYXR0cgogICAgbl9lZGdlcyA9IGJhc2VfYXR0ci5zaGFwZVswXQogICAgaWYgbl9lZGdlcyA9PSAwOgogICAgICAgIHJldHVybiBbXQoKICAgIGlmIHRhcmdldF9jbGFzcyBpcyBOb25lOgogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICB0YXJnZXRfY2xhc3MgPSBpbnQobW9kZWwod2luZG93LCBwYWlyLCBwYWlyX2F0dHIpLmFyZ21heChkaW09LTEpLml0ZW0oKSkKCiAgICBtYXNrID0gdG9yY2guemVyb3Mobl9lZGdlcywgZGV2aWNlPWRldmljZSwgcmVxdWlyZXNfZ3JhZD1UcnVlKQogICAgb3B0ID0gdG9yY2gub3B0aW0uQWRhbShbbWFza10sIGxyPWxyKQogICAgZm9yIF8gaW4gcmFuZ2UoZXBvY2hzKToKICAgICAgICBvcHQuemVyb19ncmFkKCkKICAgICAgICB3ID0gX2Nsb25lX3dpbmRvdyh3aW5kb3cpCiAgICAgICAgbSA9IHRvcmNoLnNpZ21vaWQobWFzaykudW5zcXVlZXplKDEpCiAgICAgICAgd1stMV1bUkVMX1NOQVBdLmVkZ2VfYXR0ciA9IGJhc2VfYXR0ciAqIG0KICAgICAgICBsb2dwID0gRi5sb2dfc29mdG1heChtb2RlbCh3LCBwYWlyLCBwYWlyX2F0dHIpLCBkaW09LTEpWzAsIHRhcmdldF9jbGFzc10KICAgICAgICBsb3NzID0gLWxvZ3AgKyBjb2VmZl9zaXplICogdG9yY2guc2lnbW9pZChtYXNrKS5tZWFuKCkKICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICBvcHQuc3RlcCgpCiAgICByZXR1cm4gdG9yY2guc2lnbW9pZChtYXNrKS5kZXRhY2goKS5jcHUoKS50b2xpc3QoKQo=",
"ml/train.py": "IiIiVHJhaW5pbmcgZW50cnlwb2ludCAocGxhbnMvMDMgwqcyLjQsIMKnNCkuIENocm9ub2xvZ2ljYWwgc3BsaXRzLCBwZXItZXBvY2ggbmVnYXRpdmUKcmVzYW1wbGluZyAodHJhaW4pIHZzIGZyb3plbiBuZWdhdGl2ZXMgKHZhbC90ZXN0KSwgY2xhc3Mtd2VpZ2h0ZWQvZm9jYWwgbG9zcywgZWFybHkgc3RvcHBpbmcKb24gdmFsIG1hY3JvLUYxLCB0ZW1wZXJhdHVyZSBjYWxpYnJhdGlvbiBvbiB2YWwsIHRlc3QgcmVwb3J0LCBhbmQgVyZCIHRyYWNraW5nICsgYXJ0aWZhY3RzLgoKQ2hlY2twb2ludCBwb2xpY3kgKHBsYW5zLzAzIMKnNC40KToKLSBiZXN0LnB0ICDigJQgd3JpdHRlbiBPTkxZIHdoZW4gdmFsIG1hY3JvLUYxIHN0cmljdGx5IGltcHJvdmVzICg+IGJlc3QgKyAxZS00KS4KLSBsYXN0LnB0ICDigJQgb3ZlcndyaXR0ZW4gZXZlcnkgZXBvY2ggKHJlc3VtZSBhZnRlciBhIGN1dC1vZmYgS2FnZ2xlIHNlc3Npb24pLgpFYWNoIGNoZWNrcG9pbnQgY2FycmllcyB7bW9kZWxfc3RhdGUsIG9wdGltaXplcl9zdGF0ZSwgZXBvY2gsIGJlc3RfbWFjcm9fZjEsIHByZXByb2Nlc3MsCmNhbGlicmF0b3IsIGNvbmZpZywgZ2l0X3NoYX0gc28gaXQgYm90aCByZXN1bWVzIHRyYWluaW5nIGFuZCBzZXJ2ZXMgY29uc2lzdGVudGx5LgoKUnVuOiAgcHl0aG9uIC1tIG1sLnRyYWluIC0tZGF0YS1kaXIgZGF0YXNldF9wYXJxdWV0IC0tYXJ0aWZhY3RzLWRpciBhcnRpZmFjdHMKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByYW5kb20KaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRpbWUKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKCnRyeToKICAgIGZyb20gdHFkbS5hdXRvIGltcG9ydCB0cWRtCmV4Y2VwdCBFeGNlcHRpb246ICAjIHRxZG0gaXMgb3B0aW9uYWwg4oCUIGZhbGwgYmFjayB0byBhIG5vLW9wIHNvIHRyYWluaW5nIHN0aWxsIHJ1bnMKICAgIGNsYXNzIHRxZG06ICAjIHR5cGU6IGlnbm9yZQogICAgICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpdGVyYWJsZT1Ob25lLCAqKmt3YXJncyk6CiAgICAgICAgICAgIHNlbGYuaXRlcmFibGUgPSBbXSBpZiBpdGVyYWJsZSBpcyBOb25lIGVsc2UgaXRlcmFibGUKCiAgICAgICAgZGVmIF9faXRlcl9fKHNlbGYpOgogICAgICAgICAgICByZXR1cm4gaXRlcihzZWxmLml0ZXJhYmxlKQoKICAgICAgICBkZWYgc2V0X3Bvc3RmaXgoc2VsZiwgKmEsICoqayk6CiAgICAgICAgICAgIHBhc3MKCiAgICAgICAgZGVmIGNsb3NlKHNlbGYpOgogICAgICAgICAgICBwYXNzCgpmcm9tIC5jYWxpYnJhdGUgaW1wb3J0IFRlbXBlcmF0dXJlU2NhbGVyCmZyb20gLmNvbmZpZyBpbXBvcnQgQ29uZmlnCmZyb20gLmRhdGFzZXQgaW1wb3J0IEdlb3BvbGl0aWNEYXRhc2V0LCBDCmZyb20gLmxvc3NlcyBpbXBvcnQgaW52ZXJzZV9mcmVxdWVuY3lfd2VpZ2h0cywgbWFrZV9sb3NzCmZyb20gLm1ldHJpY3MgaW1wb3J0IEV2YWx1YXRvcgpmcm9tIC5tb2RlbCBpbXBvcnQgU3BhdGlvVGVtcG9yYWxFZGdlQ2xhc3NpZmllcgoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHV0aWxpdGllcwpkZWYgc2V0X3NlZWQoc2VlZDogaW50KSAtPiBOb25lOgogICAgcmFuZG9tLnNlZWQoc2VlZCk7IG5wLnJhbmRvbS5zZWVkKHNlZWQpOyB0b3JjaC5tYW51YWxfc2VlZChzZWVkKQogICAgdG9yY2guY3VkYS5tYW51YWxfc2VlZF9hbGwoc2VlZCkKCgpkZWYgcGlja19kZXZpY2UoY2ZnOiBDb25maWcpIC0+IHRvcmNoLmRldmljZToKICAgIGlmIGNmZy5kZXZpY2UgIT0gImF1dG8iOgogICAgICAgIHJldHVybiB0b3JjaC5kZXZpY2UoY2ZnLmRldmljZSkKICAgIHJldHVybiB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKCgpkZWYgZ2l0X3NoYSgpIC0+IHN0cjoKICAgIHRyeToKICAgICAgICByZXR1cm4gc3VicHJvY2Vzcy5jaGVja19vdXRwdXQoWyJnaXQiLCAicmV2LXBhcnNlIiwgIi0tc2hvcnQiLCAiSEVBRCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRlcnI9c3VicHJvY2Vzcy5ERVZOVUxMKS5kZWNvZGUoKS5zdHJpcCgpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiAidW5rbm93biIKCgpkZWYgbW92ZV93aW5kb3cod2luZG93OiBsaXN0LCBkZXZpY2U6IHRvcmNoLmRldmljZSkgLT4gbGlzdDoKICAgIHJldHVybiBbZC50byhkZXZpY2UpIGZvciBkIGluIHdpbmRvd10KCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBjb2xsZWN0X2xvZ2l0cyhtb2RlbCwgZHMsIHNhbXBsZXMsIGRldmljZSwgZGVzYzogc3RyIHwgTm9uZSA9IE5vbmUpOgogICAgbW9kZWwuZXZhbCgpCiAgICBsb2dpdHNfYWxsLCBsYWJlbHNfYWxsID0gW10sIFtdCiAgICBpdGVyYXRvciA9IHRxZG0oc2FtcGxlcywgZGVzYz1kZXNjLCBsZWF2ZT1GYWxzZSwgdW5pdD0ibW9udGgiKSBpZiBkZXNjIGVsc2Ugc2FtcGxlcwogICAgZm9yIHMgaW4gaXRlcmF0b3I6CiAgICAgICAgd2luZG93ID0gbW92ZV93aW5kb3coZHMuYnVpbGRfd2luZG93KHMudGFyZ2V0X3RzKSwgZGV2aWNlKQogICAgICAgIGxvZ2l0cyA9IG1vZGVsKHdpbmRvdywgcy5wYWlyX2luZGV4LnRvKGRldmljZSksIHMucGFpcl9hdHRyLnRvKGRldmljZSkpCiAgICAgICAgbG9naXRzX2FsbC5hcHBlbmQobG9naXRzLmNwdSgpKTsgbGFiZWxzX2FsbC5hcHBlbmQocy5sYWJlbHMuY3B1KCkpCiAgICBpZiBub3QgbG9naXRzX2FsbDoKICAgICAgICByZXR1cm4gdG9yY2guemVyb3MoKDAsIGxlbihkcy5jZmcuY2xhc3NfbmFtZXMpKSksIHRvcmNoLnplcm9zKCgwLCksIGR0eXBlPXRvcmNoLmxvbmcpCiAgICByZXR1cm4gdG9yY2guY2F0KGxvZ2l0c19hbGwpLCB0b3JjaC5jYXQobGFiZWxzX2FsbCkKCgpkZWYgZXZhbHVhdGUobW9kZWwsIGRzLCBzYW1wbGVzLCBkZXZpY2UsIGNhbGlicmF0b3I6IFRlbXBlcmF0dXJlU2NhbGVyIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgICAgICBkZXNjOiBzdHIgfCBOb25lID0gTm9uZSk6CiAgICBsb2dpdHMsIGxhYmVscyA9IGNvbGxlY3RfbG9naXRzKG1vZGVsLCBkcywgc2FtcGxlcywgZGV2aWNlLCBkZXNjPWRlc2MpCiAgICBwcm9icyA9IGNhbGlicmF0b3IucHJvYnMobG9naXRzKSBpZiBjYWxpYnJhdG9yIGlzIG5vdCBOb25lIGVsc2UgdG9yY2guc29mdG1heChsb2dpdHMsIGRpbT0tMSkKICAgIGV2ID0gRXZhbHVhdG9yKCJjcHUiKTsKICAgIGlmIGxhYmVscy5udW1lbCgpOgogICAgICAgIGV2LnVwZGF0ZShwcm9icywgbGFiZWxzKQogICAgcmV0dXJuIGV2LmNvbXB1dGUoKSwgbG9naXRzLCBsYWJlbHMKCgpkZWYgc2F2ZV9jaGVja3BvaW50KHBhdGgsIG1vZGVsLCBvcHRpbWl6ZXIsIGVwb2NoLCBiZXN0LCBjZmcsIHBwLCBjYWxpYnJhdG9yX3RlbXA9Tm9uZSk6CiAgICB0b3JjaC5zYXZlKHsKICAgICAgICAibW9kZWxfc3RhdGUiOiBtb2RlbC5zdGF0ZV9kaWN0KCksCiAgICAgICAgIm9wdGltaXplcl9zdGF0ZSI6IG9wdGltaXplci5zdGF0ZV9kaWN0KCksCiAgICAgICAgImVwb2NoIjogZXBvY2gsCiAgICAgICAgImJlc3RfbWFjcm9fZjEiOiBiZXN0LAogICAgICAgICJwcmVwcm9jZXNzIjogcHAsCiAgICAgICAgImNhbGlicmF0b3JfdGVtcGVyYXR1cmUiOiBjYWxpYnJhdG9yX3RlbXAsCiAgICAgICAgImNvbmZpZyI6IGNmZy50b19kaWN0KCksCiAgICAgICAgImdpdF9zaGEiOiBnaXRfc2hhKCksCiAgICAgICAgImNsYXNzX25hbWVzIjogbGlzdChjZmcuY2xhc3NfbmFtZXMpLAogICAgfSwgcGF0aCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gVyZCIHdyYXBwZXIKY2xhc3MgV2FuZGJSdW46CiAgICAiIiJUaGluIHdyYXBwZXI6IG5vLW9wcyBjbGVhbmx5IGlmIHdhbmRiIGlzIG1pc3Npbmcgb3IgbW9kZT0nZGlzYWJsZWQnLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjZmc6IENvbmZpZyk6CiAgICAgICAgc2VsZi5ydW4gPSBOb25lCiAgICAgICAgaWYgY2ZnLndhbmRiX21vZGUgPT0gImRpc2FibGVkIjoKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgdHJ5OgogICAgICAgICAgICBpbXBvcnQgd2FuZGIKICAgICAgICAgICAgc2VsZi5fd2FuZGIgPSB3YW5kYgogICAgICAgICAgICBrZXkgPSBvcy5lbnZpcm9uLmdldCgiV0FOREJfQVBJX0tFWSIpCiAgICAgICAgICAgIGlmIG5vdCBrZXk6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgZnJvbSBrYWdnbGVfc2VjcmV0cyBpbXBvcnQgVXNlclNlY3JldHNDbGllbnQKICAgICAgICAgICAgICAgICAgICBrZXkgPSBVc2VyU2VjcmV0c0NsaWVudCgpLmdldF9zZWNyZXQoIldBTkRCX0FQSV9LRVkiKQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICBrZXkgPSBOb25lCiAgICAgICAgICAgIGlmIGtleToKICAgICAgICAgICAgICAgIHdhbmRiLmxvZ2luKGtleT1rZXkpCiAgICAgICAgICAgIHNlbGYucnVuID0gd2FuZGIuaW5pdCgKICAgICAgICAgICAgICAgIHByb2plY3Q9Y2ZnLndhbmRiX3Byb2plY3QsIGVudGl0eT1jZmcud2FuZGJfZW50aXR5LCBtb2RlPWNmZy53YW5kYl9tb2RlLAogICAgICAgICAgICAgICAgbmFtZT0oY2ZnLnJ1bl9uYW1lIG9yIE5vbmUpLCBjb25maWc9Y2ZnLnRvX2RpY3QoKSwgam9iX3R5cGU9InRyYWluIiwKICAgICAgICAgICAgKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICMgbmV2ZXIgbGV0IHRyYWNraW5nIGJyZWFrIHRyYWluaW5nCiAgICAgICAgICAgIHByaW50KGYiW3dhbmRiXSBkaXNhYmxlZCAoe2V9KSIpCiAgICAgICAgICAgIHNlbGYucnVuID0gTm9uZQoKICAgIGRlZiBsb2coc2VsZiwgZGF0YTogZGljdCwgc3RlcDogaW50IHwgTm9uZSA9IE5vbmUpIC0+IE5vbmU6CiAgICAgICAgaWYgc2VsZi5ydW4gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHNlbGYuX3dhbmRiLmxvZyhkYXRhLCBzdGVwPXN0ZXApCgogICAgZGVmIGNvbmZ1c2lvbihzZWxmLCB5X3RydWUsIHByZWRzLCBjbGFzc19uYW1lcyk6CiAgICAgICAgaWYgc2VsZi5ydW4gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHNlbGYuX3dhbmRiLmxvZyh7InRlc3QvY29uZnVzaW9uIjogc2VsZi5fd2FuZGIucGxvdC5jb25mdXNpb25fbWF0cml4KAogICAgICAgICAgICAgICAgICAgIHlfdHJ1ZT15X3RydWUsIHByZWRzPXByZWRzLCBjbGFzc19uYW1lcz1jbGFzc19uYW1lcyl9KQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgcGFzcwoKICAgIGRlZiBsb2dfYXJ0aWZhY3Qoc2VsZiwgZmlsZXM6IGxpc3Rbc3RyXSwgbmFtZTogc3RyLCBtZXRhZGF0YTogZGljdCwgYWxpYXNlczogbGlzdFtzdHJdKSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYucnVuIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybgogICAgICAgIHRyeToKICAgICAgICAgICAgYXJ0ID0gc2VsZi5fd2FuZGIuQXJ0aWZhY3QobmFtZSwgdHlwZT0ibW9kZWwiLCBtZXRhZGF0YT1tZXRhZGF0YSkKICAgICAgICAgICAgZm9yIGYgaW4gZmlsZXM6CiAgICAgICAgICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhmKToKICAgICAgICAgICAgICAgICAgICBhcnQuYWRkX2ZpbGUoZikKICAgICAgICAgICAgc2VsZi5ydW4ubG9nX2FydGlmYWN0KGFydCwgYWxpYXNlcz1hbGlhc2VzKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcHJpbnQoZiJbd2FuZGJdIGFydGlmYWN0IGxvZyBmYWlsZWQgKHtlfSkiKQoKICAgIGRlZiBmaW5pc2goc2VsZikgLT4gTm9uZToKICAgICAgICBpZiBzZWxmLnJ1biBpcyBub3QgTm9uZToKICAgICAgICAgICAgc2VsZi5ydW4uZmluaXNoKCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gdHJhaW4KZGVmIHRyYWluKGNmZzogQ29uZmlnKSAtPiBkaWN0OgogICAgc2V0X3NlZWQoY2ZnLnNlZWQpCiAgICBkZXZpY2UgPSBwaWNrX2RldmljZShjZmcpCiAgICBwcmludChmIlt0cmFpbl0gZGV2aWNlPXtkZXZpY2V9IGRhdGFfZGlyPXtjZmcuZGF0YV9kaXJ9IikKCiAgICBkcyA9IEdlb3BvbGl0aWNEYXRhc2V0LmZyb21fcGFycXVldChjZmcpCiAgICBvcy5tYWtlZGlycyhjZmcuYXJ0aWZhY3RzX2RpciwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBwX3BhdGggPSBvcy5wYXRoLmpvaW4oY2ZnLmFydGlmYWN0c19kaXIsICJwcmVwcm9jZXNzLnBrbCIpCiAgICBkcy5wcC5zYXZlKHBwX3BhdGgpCiAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKGNmZy5hcnRpZmFjdHNfZGlyLCAibm9kZV9pbmRleC5qc29uIiksICJ3IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAoeyJjb3VudHJ5X2lkcyI6IGRzLnBwLmNvdW50cnlfaWRzLCAiYWN0b3JfaWRzIjogZHMucHAuYWN0b3JfaWRzLAogICAgICAgICAgICAgICAgICAgImNsYXNzX25hbWVzIjogY2ZnLmNsYXNzX25hbWVzfSwgZikKCiAgICBtb2RlbCA9IFNwYXRpb1RlbXBvcmFsRWRnZUNsYXNzaWZpZXIuZnJvbV9kYXRhc2V0KGRzLCBjZmcpLnRvKGRldmljZSkKICAgIGNvdW50cyA9IGRzLmNsYXNzX2NvdW50cygidHJhaW4iLCBzZWVkPWNmZy5zZWVkKQogICAgd2VpZ2h0cyA9IGludmVyc2VfZnJlcXVlbmN5X3dlaWdodHMoY291bnRzKS50byhkZXZpY2UpCiAgICBwcmludChmIlt0cmFpbl0gY2xhc3MgY291bnRzPXtjb3VudHMudG9saXN0KCl9IHdlaWdodHM9e1tyb3VuZCh3LDIpIGZvciB3IGluIHdlaWdodHMudG9saXN0KCldfSIpCiAgICBjcml0ZXJpb24gPSBtYWtlX2xvc3MoY2ZnLmxvc3MsIHdlaWdodHMsIGNmZy5mb2NhbF9nYW1tYSkKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW1XKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9Y2ZnLmxyLCB3ZWlnaHRfZGVjYXk9Y2ZnLndlaWdodF9kZWNheSkKCiAgICB2YWxfc2FtcGxlcyA9IGRzLm1ha2Vfc2FtcGxlcygidmFsIiwgbnAucmFuZG9tLmRlZmF1bHRfcm5nKGNmZy5zZWVkICsgMSkpICAjIGZyb3plbgogICAgYmVzdF9wYXRoID0gb3MucGF0aC5qb2luKGNmZy5hcnRpZmFjdHNfZGlyLCAiYmVzdC5wdCIpCiAgICBsYXN0X3BhdGggPSBvcy5wYXRoLmpvaW4oY2ZnLmFydGlmYWN0c19kaXIsICJsYXN0LnB0IikKICAgIHdiID0gV2FuZGJSdW4oY2ZnKQoKICAgIGJlc3QsIGJlc3RfZXBvY2gsIHBhdGllbmNlID0gLTEuMCwgLTEsIDAKICAgIGZvciBlcG9jaCBpbiByYW5nZShjZmcuZXBvY2hzKToKICAgICAgICBtb2RlbC50cmFpbigpCiAgICAgICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKGNmZy5zZWVkICogMTAwMCArIGVwb2NoKSAgICMgZnJlc2ggdHJhaW4gbmVnYXRpdmVzIGVhY2ggZXBvY2gKICAgICAgICB0cmFpbl9zYW1wbGVzID0gZHMubWFrZV9zYW1wbGVzKCJ0cmFpbiIsIHJuZykKICAgICAgICBvcmRlciA9IGxpc3QocmFuZ2UobGVuKHRyYWluX3NhbXBsZXMpKSk7IHJuZy5zaHVmZmxlKG9yZGVyKQogICAgICAgIHQwID0gdGltZS50aW1lKCk7IHRvdGFsLCBuYiA9IDAuMCwgMAogICAgICAgIHBiYXIgPSB0cWRtKG9yZGVyLCBkZXNjPWYiZXBvY2gge2Vwb2NoOjAyZH0ve2NmZy5lcG9jaHMgLSAxfSIsIGxlYXZlPUZhbHNlLCB1bml0PSJiYXRjaCIpCiAgICAgICAgZm9yIGkgaW4gcGJhcjoKICAgICAgICAgICAgcyA9IHRyYWluX3NhbXBsZXNbaV0KICAgICAgICAgICAgd2luZG93ID0gbW92ZV93aW5kb3coZHMuYnVpbGRfd2luZG93KHMudGFyZ2V0X3RzKSwgZGV2aWNlKQogICAgICAgICAgICBsb2dpdHMgPSBtb2RlbCh3aW5kb3csIHMucGFpcl9pbmRleC50byhkZXZpY2UpLCBzLnBhaXJfYXR0ci50byhkZXZpY2UpKQogICAgICAgICAgICBsb3NzID0gY3JpdGVyaW9uKGxvZ2l0cywgcy5sYWJlbHMudG8oZGV2aWNlKSkKICAgICAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZCgpOyBsb3NzLmJhY2t3YXJkKCk7IG9wdGltaXplci5zdGVwKCkKICAgICAgICAgICAgdG90YWwgKz0gZmxvYXQobG9zcy5pdGVtKCkpOyBuYiArPSAxCiAgICAgICAgICAgIHBiYXIuc2V0X3Bvc3RmaXgobG9zcz1mInt0b3RhbCAvIG5iOi40Zn0iLCBiZXN0X2YxPWYie21heChiZXN0LCAwLjApOi4zZn0iKQogICAgICAgIHBiYXIuY2xvc2UoKQogICAgICAgIHRyYWluX2xvc3MgPSB0b3RhbCAvIG1heCgxLCBuYikKCiAgICAgICAgdmFsX3JlcywgXywgXyA9IGV2YWx1YXRlKG1vZGVsLCBkcywgdmFsX3NhbXBsZXMsIGRldmljZSkKICAgICAgICBkdCA9IHRpbWUudGltZSgpIC0gdDAKICAgICAgICBwcmludChmIltlcG9jaCB7ZXBvY2g6MDJkfV0gbG9zcz17dHJhaW5fbG9zczouNGZ9IHZhbF9tYWNyb0YxPXt2YWxfcmVzLm1hY3JvX2YxOi40Zn0gIgogICAgICAgICAgICAgIGYidmFsX2VjZT17dmFsX3Jlcy5lY2U6LjRmfSAoe2R0Oi4xZn1zKSIpCiAgICAgICAgbG9nID0geyJlcG9jaCI6IGVwb2NoLCAidHJhaW4vbG9zcyI6IHRyYWluX2xvc3MsICJ2YWwvbWFjcm9fZjEiOiB2YWxfcmVzLm1hY3JvX2YxLAogICAgICAgICAgICAgICAidmFsL2VjZSI6IHZhbF9yZXMuZWNlfQogICAgICAgIGZvciBjLCBmMSBpbiB6aXAoY2ZnLmNsYXNzX25hbWVzLCB2YWxfcmVzLnBlcl9jbGFzc19mMSk6CiAgICAgICAgICAgIGxvZ1tmInZhbC9mMV97Y30iXSA9IGYxCiAgICAgICAgd2IubG9nKGxvZywgc3RlcD1lcG9jaCkKCiAgICAgICAgc2F2ZV9jaGVja3BvaW50KGxhc3RfcGF0aCwgbW9kZWwsIG9wdGltaXplciwgZXBvY2gsIGJlc3QsIGNmZywgZHMucHApICAjIGV2ZXJ5IGVwb2NoCiAgICAgICAgaWYgdmFsX3Jlcy5tYWNyb19mMSA+IGJlc3QgKyAxZS00OiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIHN0cmljdCBpbXByb3ZlbWVudCBvbmx5CiAgICAgICAgICAgIGJlc3QsIGJlc3RfZXBvY2gsIHBhdGllbmNlID0gdmFsX3Jlcy5tYWNyb19mMSwgZXBvY2gsIDAKICAgICAgICAgICAgc2F2ZV9jaGVja3BvaW50KGJlc3RfcGF0aCwgbW9kZWwsIG9wdGltaXplciwgZXBvY2gsIGJlc3QsIGNmZywgZHMucHApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcGF0aWVuY2UgKz0gMQogICAgICAgICAgICBpZiBwYXRpZW5jZSA+PSBjZmcucGF0aWVuY2U6CiAgICAgICAgICAgICAgICBwcmludChmIlt0cmFpbl0gZWFybHkgc3RvcCBhdCBlcG9jaCB7ZXBvY2h9IChiZXN0PXtiZXN0Oi40Zn0gQCB7YmVzdF9lcG9jaH0pIikKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgIyAtLS0tIGxvYWQgYmVzdCwgY2FsaWJyYXRlIG9uIHZhbCwgZXZhbHVhdGUgdGVzdCAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgaWYgb3MucGF0aC5leGlzdHMoYmVzdF9wYXRoKToKICAgICAgICAjIHdlaWdodHNfb25seT1GYWxzZTogb3VyIGNoZWNrcG9pbnRzIGVtYmVkIHRoZSBQcmVwcm9jZXNzIGJ1bmRsZSAodHJ1c3RlZCwgc2VsZi13cml0dGVuKS4KICAgICAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QodG9yY2gubG9hZChiZXN0X3BhdGgsIG1hcF9sb2NhdGlvbj1kZXZpY2UsIHdlaWdodHNfb25seT1GYWxzZSlbIm1vZGVsX3N0YXRlIl0pCgogICAgXywgdmFsX2xvZ2l0cywgdmFsX2xhYmVscyA9IGV2YWx1YXRlKG1vZGVsLCBkcywgdmFsX3NhbXBsZXMsIGRldmljZSwgZGVzYz0iY2FsaWJyYXRlICh2YWwpIikKICAgIGNhbGlicmF0b3IgPSBUZW1wZXJhdHVyZVNjYWxlcigpCiAgICBpZiB2YWxfbGFiZWxzLm51bWVsKCk6CiAgICAgICAgY2FsaWJyYXRvci5maXQodmFsX2xvZ2l0cywgdmFsX2xhYmVscykKICAgIGNhbF9wYXRoID0gb3MucGF0aC5qb2luKGNmZy5hcnRpZmFjdHNfZGlyLCAiY2FsaWJyYXRvci5wa2wiKQogICAgY2FsaWJyYXRvci5zYXZlKGNhbF9wYXRoKQogICAgc2F2ZV9jaGVja3BvaW50KGJlc3RfcGF0aCwgbW9kZWwsIG9wdGltaXplciwgYmVzdF9lcG9jaCwgYmVzdCwgY2ZnLCBkcy5wcCwgY2FsaWJyYXRvci50ZW1wZXJhdHVyZSkKCiAgICB0ZXN0X3NhbXBsZXMgPSBkcy5tYWtlX3NhbXBsZXMoInRlc3QiLCBucC5yYW5kb20uZGVmYXVsdF9ybmcoY2ZnLnNlZWQgKyAyKSkKICAgIHRlc3RfcmVzX3VuY2FsLCB0ZXN0X2xvZ2l0cywgdGVzdF9sYWJlbHMgPSBldmFsdWF0ZShtb2RlbCwgZHMsIHRlc3Rfc2FtcGxlcywgZGV2aWNlLCBkZXNjPSJ0ZXN0IikKICAgIHRlc3RfcmVzX2NhbCwgXywgXyA9IGV2YWx1YXRlKG1vZGVsLCBkcywgdGVzdF9zYW1wbGVzLCBkZXZpY2UsIGNhbGlicmF0b3IpCgogICAgbWV0cmljcyA9IHsKICAgICAgICAiYmVzdF92YWxfbWFjcm9fZjEiOiBiZXN0LCAiYmVzdF9lcG9jaCI6IGJlc3RfZXBvY2gsCiAgICAgICAgInRlbXBlcmF0dXJlIjogY2FsaWJyYXRvci50ZW1wZXJhdHVyZSwKICAgICAgICAidGVzdF9tYWNyb19mMSI6IHRlc3RfcmVzX2NhbC5tYWNyb19mMSwKICAgICAgICAidGVzdF9wZXJfY2xhc3NfZjEiOiBkaWN0KHppcChjZmcuY2xhc3NfbmFtZXMsIHRlc3RfcmVzX2NhbC5wZXJfY2xhc3NfZjEpKSwKICAgICAgICAidGVzdF9lY2VfdW5jYWxpYnJhdGVkIjogdGVzdF9yZXNfdW5jYWwuZWNlLAogICAgICAgICJ0ZXN0X2VjZV9jYWxpYnJhdGVkIjogdGVzdF9yZXNfY2FsLmVjZSwKICAgICAgICAidGVzdF9jb25mdXNpb24iOiB0ZXN0X3Jlc19jYWwuY29uZnVzaW9uLAogICAgICAgICJ0ZXN0X24iOiB0ZXN0X3Jlc19jYWwubiwKICAgICAgICAiZ2l0X3NoYSI6IGdpdF9zaGEoKSwKICAgIH0KICAgIG1ldHJpY3NfcGF0aCA9IG9zLnBhdGguam9pbihjZmcuYXJ0aWZhY3RzX2RpciwgIm1ldHJpY3MuanNvbiIpCiAgICB3aXRoIG9wZW4obWV0cmljc19wYXRoLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKG1ldHJpY3MsIGYsIGluZGVudD0yKQogICAgcHJpbnQoZiJbdHJhaW5dIFRFU1QgbWFjcm8tRjE9e3Rlc3RfcmVzX2NhbC5tYWNyb19mMTouNGZ9ICIKICAgICAgICAgIGYiRUNFIHt0ZXN0X3Jlc191bmNhbC5lY2U6LjRmfS0+e3Rlc3RfcmVzX2NhbC5lY2U6LjRmfSBUPXtjYWxpYnJhdG9yLnRlbXBlcmF0dXJlOi4zZn0iKQoKICAgIGlmIHRlc3RfbGFiZWxzLm51bWVsKCk6CiAgICAgICAgcHJvYnMgPSBjYWxpYnJhdG9yLnByb2JzKHRlc3RfbG9naXRzKQogICAgICAgIHdiLmNvbmZ1c2lvbih0ZXN0X2xhYmVscy50b2xpc3QoKSwgcHJvYnMuYXJnbWF4KC0xKS50b2xpc3QoKSwgY2ZnLmNsYXNzX25hbWVzKQogICAgd2IubG9nX2FydGlmYWN0KAogICAgICAgIFtiZXN0X3BhdGgsIHBwX3BhdGgsIGNhbF9wYXRoLCBtZXRyaWNzX3BhdGhdLAogICAgICAgIG5hbWU9ZiJ7Y2ZnLndhbmRiX3Byb2plY3R9LW1vZGVsIiwKICAgICAgICBtZXRhZGF0YT17InRlc3RfbWFjcm9fZjEiOiB0ZXN0X3Jlc19jYWwubWFjcm9fZjEsICJiZXN0X3ZhbF9tYWNyb19mMSI6IGJlc3QsICJnaXRfc2hhIjogZ2l0X3NoYSgpfSwKICAgICAgICBhbGlhc2VzPVsiYmVzdCIsICJsYXRlc3QiLCBmImYxLXt0ZXN0X3Jlc19jYWwubWFjcm9fZjE6LjNmfSJdLAogICAgKQogICAgd2IuZmluaXNoKCkKICAgIHJldHVybiBtZXRyaWNzCgoKZGVmIGJ1aWxkX2FyZ3BhcnNlcigpIC0+IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyOgogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJUcmFpbiB0aGUgZ2VvcG9saXRpYyBzcGF0aW8tdGVtcG9yYWwgR05OLiIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1kYXRhLWRpciIsIGRlZmF1bHQ9Tm9uZSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWFydGlmYWN0cy1kaXIiLCBkZWZhdWx0PU5vbmUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbG9zcyIsIGRlZmF1bHQ9Tm9uZSwgY2hvaWNlcz1bIndlaWdodGVkX2NlIiwgImZvY2FsIl0pCiAgICBwLmFkZF9hcmd1bWVudCgiLS1kZXZpY2UiLCBkZWZhdWx0PU5vbmUpCiAgICBwLmFkZF9hcmd1bWVudCgiLS13YW5kYi1tb2RlIiwgZGVmYXVsdD1Ob25lLCBjaG9pY2VzPVsib25saW5lIiwgIm9mZmxpbmUiLCAiZGlzYWJsZWQiXSkKICAgIHJldHVybiBwCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXJncyA9IGJ1aWxkX2FyZ3BhcnNlcigpLnBhcnNlX2FyZ3MoKQogICAgY2ZnID0gQ29uZmlnLmZyb21fZW52KCkKICAgIGlmIGFyZ3MuZGF0YV9kaXI6IGNmZy5kYXRhX2RpciA9IGFyZ3MuZGF0YV9kaXIKICAgIGlmIGFyZ3MuYXJ0aWZhY3RzX2RpcjogY2ZnLmFydGlmYWN0c19kaXIgPSBhcmdzLmFydGlmYWN0c19kaXIKICAgIGlmIGFyZ3MuZXBvY2hzIGlzIG5vdCBOb25lOiBjZmcuZXBvY2hzID0gYXJncy5lcG9jaHMKICAgIGlmIGFyZ3MubG9zczogY2ZnLmxvc3MgPSBhcmdzLmxvc3MKICAgIGlmIGFyZ3MuZGV2aWNlOiBjZmcuZGV2aWNlID0gYXJncy5kZXZpY2UKICAgIGlmIGFyZ3Mud2FuZGJfbW9kZTogY2ZnLndhbmRiX21vZGUgPSBhcmdzLndhbmRiX21vZGUKICAgIHRyYWluKGNmZykKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
"ml/infer.py": "IiIiSW5mZXJlbmNlIChwbGFucy8wMyDCp01vZGVsIElucHV0L091dHB1dCwgwqczKS4gTG9hZHMgdGhlIHRyYWluZWQgbW9kZWwgKyB0aGUgcGVyc2lzdGVkCnByZXByb2Nlc3MgYnVuZGxlICsgdGVtcGVyYXR1cmUgY2FsaWJyYXRvciwgcmVidWlsZHMgdGhlIGxpdmUgdGVtcG9yYWwgd2luZG93IGZyb20gdGhlCmFnZ3JlZ2F0ZWQgUGFycXVldCBleHBvcnQsIGFuZCBwcmVkaWN0cyB0aGUgNS1jbGFzcyBkaXN0cmlidXRpb24gZm9yIGEgZGlyZWN0ZWQgQ291bnRyeSBwYWlyCmF0IGEgdGFyZ2V0IG1vbnRoLiBSZXVzZXMgdGhlIGV4YWN0IHNhbWUgbW9kZWwucHkgLyBkYXRhc2V0LnB5IC8gZmVhdHVyZXMucHkgYXMgdHJhaW5pbmcsIHNvCnNlcnZpbmcgY2FuIG5ldmVyIGRyaWZ0IGZyb20gaG93IGl0IHdhcyB0cmFpbmVkLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKaW1wb3J0IHRvcmNoCgpmcm9tIC5jYWxpYnJhdGUgaW1wb3J0IFRlbXBlcmF0dXJlU2NhbGVyCmZyb20gLmNvbmZpZyBpbXBvcnQgQ29uZmlnCmZyb20gLmRhdGFzZXQgaW1wb3J0IEdlb3BvbGl0aWNEYXRhc2V0CmZyb20gLmV4cGxhaW4gaW1wb3J0IGdubl9leHBsYWluZXJfZWRnZV9tYXNrLCBpbnRlZ3JhdGVkX2dyYWRpZW50cwpmcm9tIC5mZWF0dXJlcyBpbXBvcnQgUHJlcHJvY2Vzcwpmcm9tIC5tb2RlbCBpbXBvcnQgU3BhdGlvVGVtcG9yYWxFZGdlQ2xhc3NpZmllcgoKCkBkYXRhY2xhc3MKY2xhc3MgUHJlZGljdGlvbjoKICAgIHNvdXJjZV9pZDogc3RyCiAgICB0YXJnZXRfaWQ6IHN0cgogICAgdGltZV9zdGVwOiBpbnQKICAgIHByb2JhYmlsaXRpZXM6IGRpY3Rbc3RyLCBmbG9hdF0KICAgIHByZWRpY3RlZF9jbGFzczogc3RyCiAgICBjb25maWRlbmNlOiBmbG9hdAoKCmNsYXNzIFByZWRpY3RvcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjZmc6IENvbmZpZywgY2hlY2twb2ludDogc3RyID0gImJlc3QucHQiKToKICAgICAgICBpbXBvcnQgb3MKICAgICAgICBzZWxmLmNmZyA9IGNmZwogICAgICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIgaWYgKGNmZy5kZXZpY2UgaW4gKCJhdXRvIiwgImN1ZGEiKSBhbmQgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSkgZWxzZSAiY3B1IikKICAgICAgICBzZWxmLmRldmljZSA9IGRldmljZQoKICAgICAgICBwcCA9IFByZXByb2Nlc3MubG9hZChvcy5wYXRoLmpvaW4oY2ZnLmFydGlmYWN0c19kaXIsICJwcmVwcm9jZXNzLnBrbCIpKQogICAgICAgIHNlbGYuZHMgPSBHZW9wb2xpdGljRGF0YXNldC5mcm9tX3BhcnF1ZXQoY2ZnLCBwcmVwcm9jZXNzPXBwKQoKICAgICAgICBja3B0X3BhdGggPSBvcy5wYXRoLmpvaW4oY2ZnLmFydGlmYWN0c19kaXIsIGNoZWNrcG9pbnQpCiAgICAgICAgIyB3ZWlnaHRzX29ubHk9RmFsc2U6IGNoZWNrcG9pbnRzIGVtYmVkIHRoZSBQcmVwcm9jZXNzIGJ1bmRsZSAodHJ1c3RlZCwgc2VsZi13cml0dGVuKS4KICAgICAgICBja3B0ID0gdG9yY2gubG9hZChja3B0X3BhdGgsIG1hcF9sb2NhdGlvbj1kZXZpY2UsIHdlaWdodHNfb25seT1GYWxzZSkKICAgICAgICBzZWxmLm1vZGVsID0gU3BhdGlvVGVtcG9yYWxFZGdlQ2xhc3NpZmllci5mcm9tX2RhdGFzZXQoc2VsZi5kcywgY2ZnKS50byhkZXZpY2UpCiAgICAgICAgc2VsZi5tb2RlbC5sb2FkX3N0YXRlX2RpY3QoY2twdFsibW9kZWxfc3RhdGUiXSkKICAgICAgICBzZWxmLm1vZGVsLmV2YWwoKQoKICAgICAgICB0ZW1wID0gY2twdC5nZXQoImNhbGlicmF0b3JfdGVtcGVyYXR1cmUiKQogICAgICAgIGlmIHRlbXAgaXMgTm9uZToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgc2VsZi5jYWxpYnJhdG9yID0gVGVtcGVyYXR1cmVTY2FsZXIubG9hZChvcy5wYXRoLmpvaW4oY2ZnLmFydGlmYWN0c19kaXIsICJjYWxpYnJhdG9yLnBrbCIpKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgc2VsZi5jYWxpYnJhdG9yID0gVGVtcGVyYXR1cmVTY2FsZXIoMS4wKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuY2FsaWJyYXRvciA9IFRlbXBlcmF0dXJlU2NhbGVyKHRlbXApCgogICAgZGVmIF9pbmRpY2VzKHNlbGYsIHNvdXJjZV9pZDogc3RyLCB0YXJnZXRfaWQ6IHN0cikgLT4gdHVwbGVbaW50LCBpbnRdOgogICAgICAgIHUgPSBzZWxmLmRzLmNvdW50cnlfaW5kZXguZ2V0KHNvdXJjZV9pZCkKICAgICAgICB2ID0gc2VsZi5kcy5jb3VudHJ5X2luZGV4LmdldCh0YXJnZXRfaWQpCiAgICAgICAgaWYgdSBpcyBOb25lIG9yIHYgaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJ1bmtub3duIGNvdW50cnkgaWQocyk6IHtzb3VyY2VfaWQhcn0ve3RhcmdldF9pZCFyfSIpCiAgICAgICAgcmV0dXJuIHUsIHYKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgcHJlZGljdChzZWxmLCBzb3VyY2VfaWQ6IHN0ciwgdGFyZ2V0X2lkOiBzdHIsIHRpbWVfc3RlcDogaW50KSAtPiBQcmVkaWN0aW9uOgogICAgICAgIHUsIHYgPSBzZWxmLl9pbmRpY2VzKHNvdXJjZV9pZCwgdGFyZ2V0X2lkKQogICAgICAgIHdpbmRvdyA9IFtkLnRvKHNlbGYuZGV2aWNlKSBmb3IgZCBpbiBzZWxmLmRzLmJ1aWxkX3dpbmRvdyh0aW1lX3N0ZXApXQogICAgICAgIHBhaXIgPSB0b3JjaC50ZW5zb3IoW1t1LCB2XV0sIGR0eXBlPXRvcmNoLmxvbmcsIGRldmljZT1zZWxmLmRldmljZSkKICAgICAgICBhdHRyID0gc2VsZi5fcGFpcl9hdHRyKHRpbWVfc3RlcCwgdSwgdikKICAgICAgICBsb2dpdHMgPSBzZWxmLm1vZGVsKHdpbmRvdywgcGFpciwgYXR0cikKICAgICAgICBwcm9icyA9IHNlbGYuY2FsaWJyYXRvci5wcm9icyhsb2dpdHMuY3B1KCkpWzBdCiAgICAgICAgbmFtZXMgPSBzZWxmLmNmZy5jbGFzc19uYW1lcwogICAgICAgIGlkeCA9IGludChwcm9icy5hcmdtYXgoKSkKICAgICAgICByZXR1cm4gUHJlZGljdGlvbigKICAgICAgICAgICAgc291cmNlX2lkPXNvdXJjZV9pZCwgdGFyZ2V0X2lkPXRhcmdldF9pZCwgdGltZV9zdGVwPXRpbWVfc3RlcCwKICAgICAgICAgICAgcHJvYmFiaWxpdGllcz17bjogZmxvYXQocCkgZm9yIG4sIHAgaW4gemlwKG5hbWVzLCBwcm9icy50b2xpc3QoKSl9LAogICAgICAgICAgICBwcmVkaWN0ZWRfY2xhc3M9bmFtZXNbaWR4XSwgY29uZmlkZW5jZT1mbG9hdChwcm9ic1tpZHhdKSwKICAgICAgICApCgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIHByZWRpY3RfYmF0Y2goc2VsZiwgcGFpcnM6IGxpc3RbdHVwbGVbc3RyLCBzdHJdXSwgdGltZV9zdGVwOiBpbnQpIC0+IGxpc3RbUHJlZGljdGlvbl06CiAgICAgICAgaWYgbm90IHBhaXJzOgogICAgICAgICAgICByZXR1cm4gW10KICAgICAgICB3aW5kb3cgPSBbZC50byhzZWxmLmRldmljZSkgZm9yIGQgaW4gc2VsZi5kcy5idWlsZF93aW5kb3codGltZV9zdGVwKV0KICAgICAgICBpZHhfcGFpcnMsIGF0dHJzID0gW10sIFtdCiAgICAgICAgZm9yIHMsIHQgaW4gcGFpcnM6CiAgICAgICAgICAgIHUsIHYgPSBzZWxmLl9pbmRpY2VzKHMsIHQpCiAgICAgICAgICAgIGlkeF9wYWlycy5hcHBlbmQoKHUsIHYpKQogICAgICAgICAgICBhdHRycy5hcHBlbmQoc2VsZi5fcGFpcl9hdHRyKHRpbWVfc3RlcCwgdSwgdikpCiAgICAgICAgcGFpciA9IHRvcmNoLnRlbnNvcihpZHhfcGFpcnMsIGR0eXBlPXRvcmNoLmxvbmcsIGRldmljZT1zZWxmLmRldmljZSkKICAgICAgICBhdHRyID0gdG9yY2guY2F0KGF0dHJzLCBkaW09MCkKICAgICAgICBwcm9icyA9IHNlbGYuY2FsaWJyYXRvci5wcm9icyhzZWxmLm1vZGVsKHdpbmRvdywgcGFpciwgYXR0cikuY3B1KCkpCiAgICAgICAgbmFtZXMgPSBzZWxmLmNmZy5jbGFzc19uYW1lcwogICAgICAgIG91dCA9IFtdCiAgICAgICAgZm9yIChzLCB0KSwgcCBpbiB6aXAocGFpcnMsIHByb2JzKToKICAgICAgICAgICAgaSA9IGludChwLmFyZ21heCgpKQogICAgICAgICAgICBvdXQuYXBwZW5kKFByZWRpY3Rpb24ocywgdCwgdGltZV9zdGVwLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAge246IGZsb2F0KHgpIGZvciBuLCB4IGluIHppcChuYW1lcywgcC50b2xpc3QoKSl9LCBuYW1lc1tpXSwgZmxvYXQocFtpXSkpKQogICAgICAgIHJldHVybiBvdXQKCiAgICBkZWYgZXhwbGFpbihzZWxmLCBzb3VyY2VfaWQ6IHN0ciwgdGFyZ2V0X2lkOiBzdHIsIHRpbWVfc3RlcDogaW50KSAtPiBkaWN0OgogICAgICAgIHUsIHYgPSBzZWxmLl9pbmRpY2VzKHNvdXJjZV9pZCwgdGFyZ2V0X2lkKQogICAgICAgIHdpbmRvdyA9IFtkLnRvKHNlbGYuZGV2aWNlKSBmb3IgZCBpbiBzZWxmLmRzLmJ1aWxkX3dpbmRvdyh0aW1lX3N0ZXApXQogICAgICAgIGF0dHIgPSBzZWxmLl9wYWlyX2F0dHIodGltZV9zdGVwLCB1LCB2KQogICAgICAgIGlnID0gaW50ZWdyYXRlZF9ncmFkaWVudHMoc2VsZi5tb2RlbCwgd2luZG93LCB1LCB2LCBhdHRyKQogICAgICAgIGVkZ2VfbWFzayA9IGdubl9leHBsYWluZXJfZWRnZV9tYXNrKHNlbGYubW9kZWwsIHdpbmRvdywgdSwgdiwgYXR0ciwgdGFyZ2V0X2NsYXNzPWlnLnRhcmdldF9jbGFzcykKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAidGFyZ2V0X2NsYXNzIjogc2VsZi5jZmcuY2xhc3NfbmFtZXNbaWcudGFyZ2V0X2NsYXNzXSwKICAgICAgICAgICAgImludGVncmF0ZWRfZ3JhZGllbnRzIjogewogICAgICAgICAgICAgICAgImVkZ2VfZmVhdHVyZV9hdHRyaWJ1dGlvbiI6IGlnLmVkZ2VfYXR0cmlidXRpb24sCiAgICAgICAgICAgICAgICAic291cmNlX25vZGVfYXR0cmlidXRpb24iOiBpZy51X25vZGVfYXR0cmlidXRpb24sCiAgICAgICAgICAgICAgICAidGFyZ2V0X25vZGVfYXR0cmlidXRpb24iOiBpZy52X25vZGVfYXR0cmlidXRpb24sCiAgICAgICAgICAgICAgICAiY29tcGxldGVuZXNzX2dhcCI6IGlnLmNvbXBsZXRlbmVzc19nYXAsCiAgICAgICAgICAgIH0sCiAgICAgICAgICAgICJnbm5fZXhwbGFpbmVyX2VkZ2VfaW1wb3J0YW5jZSI6IGVkZ2VfbWFzaywKICAgICAgICB9CgogICAgZGVmIF9wYWlyX2F0dHIoc2VsZiwgdGltZV9zdGVwOiBpbnQsIHU6IGludCwgdjogaW50KSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgbG9va3VwID0gc2VsZi5kcy5fcGFpcl9sb29rdXAodGltZV9zdGVwKQogICAgICAgIGhpdCA9IGxvb2t1cC5nZXQoKHUsIHYpKQogICAgICAgIGVkaW0gPSBzZWxmLmRzLnBwLmVkZ2VfZGltCiAgICAgICAgaWYgaGl0IGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiB0b3JjaC56ZXJvcygoMSwgZWRpbSksIGR0eXBlPXRvcmNoLmZsb2F0MzIsIGRldmljZT1zZWxmLmRldmljZSkKICAgICAgICByZXR1cm4gdG9yY2gudGVuc29yKGhpdCwgZHR5cGU9dG9yY2guZmxvYXQzMiwgZGV2aWNlPXNlbGYuZGV2aWNlKS51bnNxdWVlemUoMCkK",
"ml/__init__.py": ""
}
for _p, _b in FILES.items():
    open(_p, 'w').write(base64.b64decode(_b).decode())
print('wrote', len(FILES), 'ml package files')


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('W&B key loaded from Kaggle Secret')
except Exception as e:
    print('Add a Kaggle Secret named WANDB_API_KEY to enable logging:', e)
os.environ.setdefault('WANDB_ENTITY', 'anna-nechytailenko-kyiv-school-of-economics')
os.environ.setdefault('WANDB_PROJECT', 'geosimulation')


In [ ]:
import os
def find_data():
    for root in ['/kaggle/input', '.']:
        if not os.path.isdir(root):
            continue
        for dp, _dn, fn in os.walk(root):
            if 'node_snapshots.parquet' in fn:
                return dp
    return 'dataset_parquet'
os.environ['GEO_DATA_DIR'] = find_data()
os.environ['GEO_ARTIFACTS_DIR'] = '/kaggle/working/artifacts' if os.path.isdir('/kaggle/working') else 'artifacts'
from ml.config import Config
from ml.train import train
cfg = Config.from_env()
print('data_dir =', cfg.data_dir, '| artifacts =', cfg.artifacts_dir)
metrics = train(cfg)
metrics
